# 🎵 AI Music Detection: A Journey from Spectral Snapshots to Temporal Intelligence
**Repository:** [AI_Music_Detection](https://github.com/NikolozDarsalia/AI_Music_Detection.git)

## Executive Summary
This project investigates the detection of synthetic music (Udio, Suno) versus human-generated audio (FMA). Rather than simply chasing high accuracy, the core objective is generalizability: ensuring the model identifies "AI-ness" rather than falling for confounding factors like genre or recording quality.

### Data Strategy & Quality Control
- Source: Based on the SleepyJesse/ai_music_large dataset, specifically focusing on Udio and Suno as modern, high-fidelity benchmarks.

- Chunk-Based Training: Tracks are divided into 10-second chunks to increase the sample size and focus on local acoustic features.

Track-Level Evaluation: While the model trains on chunks, I evaluate performance using soft voting (averaging scores across a full track) to mirror real-world forensic applications.

Efficiency: To "fail fast" and iterate on architectures, experiments were conducted on a 6,000-chunk representative subset, which provided a clear signal for model comparison.

### Combatting Confounding Factors (MERT Mining)
A major risk in AI detection is the model learning "shortcuts" (e.g., distinguishing Electronic music from Classical rather than AI from Human).

I utilized MERT (Self-Supervised Music Embeddings) to perform Hard-Negative Mining (Human-generated similar to AI tracks and AI-generated similar to human tracks).

By using a 30/70 strategy (30% "Hard" samples / 70% random), I forced the model to relatively ignore easy solutions and focus on the structural nuances of the audio.

### Evolutionary Experimental Path
This notebook documents four distinct stages of development:

- MLP (Baseline): Establishing the floor of performance.

- CNN: Moving to 2D spectral feature extraction.

- CNN-Mined: Applying the MERT-mining strategy to the CNN to test for increased robustness.

- CNN-BiGRU (Final Model): My top-performing architecture. This CRNN combines local spectral extraction with temporal sequence modeling, effectively "listening" to how a sound evolves over time.

### Interpretability & "Confidently Wrong" Analysis
Beyond metrics, this project utilizes Integrated Gradients and Grad-CAM to "see" what the model hears. I focus specifically on "Confidently Wrong" tracks-samples where the model is highly certain but incorrect-to identify remaining blind spots in current AI detection technology.

### 🛠 How to Replicate
- Clone the repository and install dependencies (see pyproject.toml).

- Run 01_Data_Collection.ipynb followed by 02_EDA.ipynb.

These will generate the necessary MERT-mined manifests required for the experiments in this notebook.

# Imports

In [ ]:
import sys
import os
import logging
import importlib
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Add src/ to path
sys.path.insert(0, "src/")

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s | %(levelname)s | %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger(__name__)
log.info("Imports complete.")

In [ ]:
# Config & Device
import config
importlib.reload(config)

device = (torch.device("mps")  if torch.backends.mps.is_available() else
          torch.device("cuda") if torch.cuda.is_available() else
          torch.device("cpu"))
log.info(f"Device : {device}")
log.info(f"Root   : {config.ROOT_DIR}")
log.info(f"Data   : {config.DATA_DIR}")

# Data Preparation

## Generate All Three Splits

In [ ]:
# Run once. Saves CSVs to data/splits/
# Re-running is safe — files are overwritten.

from splits import generate_all_splits
importlib.reload(__import__("splits"))

splits = generate_all_splits(config)

# Verify
for name, df in splits.items():
    print(f"\n{name}:")
    print(df.groupby(["split", "label", "source_name"])
            .size().rename("tracks").to_string())

## Build Chunk Indices for All Splits

In [ ]:
# Generates start positions without loading audio.
# Train split: N_CHUNKS_*_TRAIN chunks per track
# Val/test   : N_CHUNKS_*_EVAL  chunks per track

from sampling import build_chunk_index, cap_ai_chunks
importlib.reload(__import__("sampling"))

chunk_indices = {}   # {strategy_name: chunk_df}

for strategy_name, split_df in splits.items():
    log.info(f"\nBuilding chunk index: {strategy_name}")

    raw_chunks = build_chunk_index(
        split_df    = split_df,
        config      = config,
        output_path = config.CHUNKS_DIR / f"{strategy_name}_chunks_raw.csv",
    )

    # Cap AI to match human in train split only
    balanced = cap_ai_chunks(raw_chunks, config, split="train")
    balanced.to_csv(
        config.CHUNKS_DIR / f"{strategy_name}_chunks_balanced.csv",
        index=False,
    )
    chunk_indices[strategy_name] = balanced

    print(f"\n{strategy_name} — final chunk counts:")
    print(balanced.groupby(["split", "label", "source_name"])
                  .size().rename("chunks").to_string())

## Generate Spectrograms for All Splits

In [ ]:
# # Most time-consuming step — runs once, then reload with load_manifest().
# # Generates separate spectrogram directories per split strategy.
# #
# # Progress is logged every 500 chunks.

# from spectrogram import generate_spectrograms
# importlib.reload(__import__("spectrogram"))

# manifests = {}   # {strategy_name: manifest_df}

# for strategy_name, chunk_df in chunk_indices.items():
#     spec_dir = config.SPECTROGRAMS_DIR / strategy_name
#     log.info(f"\nGenerating spectrograms: {strategy_name} → {spec_dir}")

#     manifest = generate_spectrograms(
#         chunk_df   = chunk_df,
#         config     = config,
#         output_dir = spec_dir,
#     )
#     manifests[strategy_name] = manifest

#     print(f"\n{strategy_name} — spectrogram summary:")
#     print(manifest.groupby(["split", "label", "source_name"])
#                   .size().rename("chunks").to_string())

# log.info("\nAll spectrograms generated.")

## Reload Manifest (use this on subsequent runs)

In [ ]:
# Skip Section 1.3 and run only this cell after the first run.

from spectrogram import load_manifest
importlib.reload(__import__("spectrogram"))

STRATEGIES = ["random", "udio_train", "suno_train"]
manifests  = {}

for strategy_name in STRATEGIES:
    spec_dir = config.SPECTROGRAMS_DIR / strategy_name
    if not (spec_dir / "manifest.csv").exists():
        log.warning(f"No manifest for {strategy_name} — run Section 1.3 first.")
        continue
    manifest, mean, std = load_manifest(spec_dir)
    manifests[strategy_name] = manifest
    log.info(f"  {strategy_name}: {len(manifest)} chunks | "
             f"norm mean={mean:.4f}, std={std:.4f}")

# Quick sanity check
strategy = "random"
sample   = np.load(manifests[strategy].iloc[0]["npy_path"])
print(f"\nSanity check ({strategy}):")
print(f"  Spec shape : {sample.shape}   (expected: {config.SPEC_SHAPE})")
print(f"  dtype      : {sample.dtype}")
print(f"  value range: [{sample.min():.3f}, {sample.max():.3f}]")

# Experiment Configuration

## Training Hyperparameters

In [ ]:
# Edit these before each experiment.
# These are passed directly to train() — config.py is not modified.

from train import TrainingConfig

TRAIN_CFG = TrainingConfig(
    learning_rate       = 1e-4,
    weight_decay        = 1e-4,
    batch_size          = 64,
    num_epochs          = 30,
    lr_patience         = 6,
    lr_factor           = 0.5,
    lr_min              = 1e-6,
    warmup_epochs       = 0,
    early_stop_patience = 10,
    scheduler_metric   = "val_loss",    # LR drops when loss stops improving
    checkpoint_metric  = "val_auroc",   # save best AUROC model
    use_augmentation    = True,
    freq_mask_param     = 15,
    time_mask_param     = 50,
    n_freq_masks        = 2,
    n_time_masks        = 2,
)

TRAIN_CFG.log_summary()

## Active Split Strategy

All of the experiments are based on 'random' strategy, 'udio_train' and 'suno_train' strategies were created for testing generalizability of the architectures, to train on Udio and test on Suno and vice versa, but due to low amount of the Suno data, this does not make much sense, and even on 'random' strategy I can evaluate how well generalizable the model is on Suno, when it knows just a few tracks of it.

In [ ]:
# Change ACTIVE_STRATEGY to switch between splits.
# All downstream cells use this variable.

ACTIVE_STRATEGY = "random"   # "random" | "udio_train" | "suno_train"

manifest = manifests[ACTIVE_STRATEGY]
log.info(f"Active strategy : {ACTIVE_STRATEGY}")
log.info(f"Manifest size   : {len(manifest)} chunks")
print(manifest.groupby(["split","label","source_name"])
              .size().rename("chunks").to_string())

## Results Tracker

In [ ]:
# Accumulates all experiment results for final comparison.
# Persists to disk — safe to re-run.

from results import load_results, add_result, show_results, compare_runs

RESULTS_PATH = config.RESULTS_DIR / "all_results.json"
results_df   = load_results(RESULTS_PATH)
log.info(f"Loaded {len(results_df)} existing results.")

# Fast Subset Loader

For quick baseline experiments, I use only 6000 chunks for training the models


In [ ]:
from torch.utils.data import DataLoader, Subset
from dataset import MelSpecDataset, _collate_fn
import platform

def build_fast_loaders(
    manifest,
    config,
    fraction:    float = 0.25,   # use 25% of data
    batch_size:  int   = 64,
    random_seed: int   = config.SEED,
) -> dict:
    """
    Build small loaders for fast baseline experiments.
    Subsets the manifest — no spectrogram regeneration needed.
    Val and test are kept at full size for honest evaluation.
    Only the train split is reduced.

    fraction=0.25 → ~6K train chunks instead of 24K → ~4x faster per epoch
    fraction=0.10 → ~2.4K train chunks → ~10x faster per epoch
    """
    rng = np.random.default_rng(random_seed)

    loaders = {}
    for split in ["train", "val", "test"]:
        split_manifest = manifest[manifest["split"] == split]
        if len(split_manifest) == 0:
            continue

        ds = MelSpecDataset(
            manifest = manifest,
            split    = split,
            config   = config,
            augment  = (split == "train"),
        )

        if split == "train":
            # Stratified subset — preserve human/AI balance
            human_idx = np.where(ds.labels == 0)[0]
            ai_idx    = np.where(ds.labels == 1)[0]

            n_human = int(len(human_idx) * fraction)
            n_ai    = int(len(ai_idx)    * fraction)

            chosen_human = rng.choice(human_idx, n_human, replace=False)
            chosen_ai    = rng.choice(ai_idx,    n_ai,    replace=False)
            chosen_idx   = np.concatenate([chosen_human, chosen_ai])
            rng.shuffle(chosen_idx)

            ds      = Subset(ds, chosen_idx.tolist())
            shuffle = True
            sampler = None
            drop_last = True
        else:
            shuffle   = False
            sampler   = None
            drop_last = False

        loaders[split] = DataLoader(
            ds,
            batch_size  = batch_size,
            shuffle     = shuffle,
            sampler     = sampler,
            num_workers = 0 if platform.system() == "Darwin" else 4,
            pin_memory  = False,
            drop_last   = drop_last,
            collate_fn  = _collate_fn,
        )

        n = len(ds)
        log.info(f"  {split:5s}: {n} chunks | "
                 f"{len(loaders[split])} batches")

    return loaders

# Experiment: MLP Baseline

## MLP Architecture

In [ ]:
class SpectrogramMLP(nn.Module):
    """
    Baseline MLP for Spectrogram Classification.
    Flattens the input (Freq x Time) and passes it through fully connected layers.
    """
    def __init__(
        self, 
        input_shape: tuple[int, int] = config.SPEC_SHAPE,  # (n_mels, time_steps)
        hidden_dims: list[int] = [512, 256, 128],
        dropout_rate: float = 0.3,
        num_classes: int = 2
    ):
        super().__init__()
        
        # Calculate the flattened input size: n_mels * time_steps
        self.input_dim = input_shape[0] * input_shape[1]
        
        layers = []
        curr_dim = self.input_dim
        
        for h_dim in hidden_dims:
            layers.append(nn.Linear(curr_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim)) # Essential for training stability
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            curr_dim = h_dim
            
        self.encoder = nn.Sequential(*layers)
        
        # Final Classification Head
        self.classifier = nn.Linear(curr_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, 1, n_mels, time_steps) or (batch, n_mels, time_steps)
        
        # 1. Flatten the spectrogram
        x = x.view(x.size(0), -1) 
        
        # 2. Pass through hidden layers
        features = self.encoder(x)
        
        # 3. Output logits
        logits = self.classifier(features)
        return logits

In [ ]:
# Flat MLP on flattened spectrograms.
# baseline — MLP with 3 hidden layers, proves the data is learnable.

mlp_model = SpectrogramMLP(
    input_shape = config.SPEC_SHAPE,
    hidden_dims = [118, 64, 32],
    dropout_rate = 0.4
).to(device)

num_params_mlp = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params_mlp:,}")

## Build DataLoaders

In [ ]:
from dataset import build_dataloaders

mlp_loaders = build_fast_loaders(
    manifest   = manifest,
    config     = config,
    fraction=0.25
)

## Train MLP

In [ ]:
model_type = 'mlp'

In [ ]:
from train import train

MLP_RUN = f"{model_type}_{ACTIVE_STRATEGY}"

mlp_history = train(
    model       = mlp_model,
    loaders     = mlp_loaders,
    train_cfg   = TRAIN_CFG,
    base_config = config,
    device      = device,
    manifest    = manifest,
    run_name    = MLP_RUN,
    model_type = model_type
)

## Evaluate MLP

In [ ]:
from train import load_checkpoint
from evaluate import (evaluate, save_evaluation_results,
                      plot_training_history, build_result_row)

load_checkpoint(
    mlp_model,
    config.CHECKPOINT_DIR / f"{MLP_RUN}_best.pt",
    device,
)

mlp_test_results = evaluate(
    model      = mlp_model,
    loader     = mlp_loaders["test"],
    device     = device,
    split_name = "test",
)

save_evaluation_results(
    results    = mlp_test_results,
    output_dir = config.LOGS_DIR,
    run_name   = MLP_RUN,
    split_name = "test",
)

plot_training_history(
    mlp_history,
    output_path = config.LOGS_DIR / f"{MLP_RUN}_curves.png",
    title       = f"{model_type.upper()} Baseline — {ACTIVE_STRATEGY}",
)

# Add to results table
results_df = add_result(
    results_df,
    build_result_row(
        run_name       = MLP_RUN,
        results        = mlp_test_results,
        model_type       = model_type,
        train_cfg      = TRAIN_CFG,
        split_strategy = ACTIVE_STRATEGY,
        extra_tags     = {"hard_negatives": False, "notes": f"baseline {model_type.upper()}"},
    ),
    RESULTS_PATH,
)
show_results(results_df)

In [ ]:
# =============================================================================
# Held-out test analysis (threshold chosen on validation)
# =============================================================================

results  = mlp_test_results   # swap to cnn_a_results etc. for other models
run_name = MLP_RUN

labels   = results["labels"]
probs    = results["probs"]
sources  = np.array(results["sources"])

# Select the decision threshold on validation data, then apply it to test.
from sklearn.metrics import f1_score
val_results = evaluate(
    model=mlp_model,
    loader=mlp_loaders["val"],
    device=device,
    split_name="val",
)
val_labels = val_results["labels"]
val_probs = val_results["probs"]
thresholds = np.linspace(0.01, 0.99, 200)
val_f1_scores = [
    f1_score(val_labels, (val_probs >= t).astype(int), zero_division=0)
    for t in thresholds
]
best_thresh = thresholds[np.argmax(val_f1_scores)]
preds = (probs >= best_thresh).astype(int)
print(f"Validation-selected threshold: {best_thresh:.3f}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"{run_name} — test set analysis (threshold from validation)", fontsize=13)

# ── 1. ROC curve ──────────────────────────────────────────────────────────────
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(labels, probs)
ax.plot(fpr, tpr, color="#4C72B0", linewidth=2,
        label=f"AUROC = {results['chunk']['auroc']:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend()
ax.grid(alpha=0.3)

# ── 2. Precision-Recall curve ────────────────────────────────────────────────
ax = axes[0, 1]
prec, rec, _ = precision_recall_curve(labels, probs)
ax.plot(rec, prec, color="#DD8452", linewidth=2,
        label=f"AUPRC = {results['chunk']['auprc']:.4f}")
ax.axhline(labels.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Baseline = {labels.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (AI = positive)")
ax.legend()
ax.grid(alpha=0.3)

# ── 3. Probability distribution ───────────────────────────────────────────────
ax = axes[0, 2]
ax.hist(probs[labels == 0], bins=50, alpha=0.6,
        color="#4C72B0", label="Human", density=True)
ax.hist(probs[labels == 1], bins=50, alpha=0.6,
        color="#DD8452", label="AI",    density=True)
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_thresh:.2f}")
ax.set_xlabel("P(AI)")
ax.set_ylabel("Density")
ax.set_title("Predicted Probability Distribution")
ax.legend()
ax.grid(alpha=0.3)

# ── 4. Confusion matrix ───────────────────────────────────────────────────────
ax = axes[1, 0]
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Human", "AI"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix (threshold={best_thresh:.2f})")


# ── 5. Per-source AUROC (pairwise) ────────────────────────────────────────────
ax = axes[1, 1]
pair_keys   = [k for k in results["per_source"]
               if k.startswith("human_vs_")]
single_keys = [k for k in results["per_source"]
               if not k.startswith("human_vs_")]

plot_keys   = pair_keys if pair_keys else single_keys
plot_labels = [k.replace("human_vs_", "FMA vs ") for k in plot_keys]
plot_aurocs = [results["per_source"][k]["chunk"]["auroc"]
               for k in plot_keys]
plot_counts = [results["per_source"][k]["track"]["n_tracks"]
               for k in plot_keys]

colors = ["#DD8452" if "Suno" in k else "#55A868" for k in plot_keys]
bars   = ax.bar(plot_labels, plot_aurocs, color=colors, alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylim(0.4, 1.05)
ax.set_ylabel("Chunk AUROC")
ax.set_title("Pairwise AUROC (FMA vs each AI source)")

for bar, count, auroc in zip(bars, plot_counts, plot_aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{auroc:.3f}\n({count} tracks)",
            ha="center", va="bottom", fontsize=9)
ax.grid(alpha=0.3, axis="y")


# ── 6. Track-level probability distribution ───────────────────────────────────
ax = axes[1, 2]
track_ids   = results["track_ids"]
unique_tids = np.unique(track_ids)

track_probs  = []
track_labels = []
for tid in unique_tids:
    mask = track_ids == tid
    track_probs.append(probs[mask].mean())
    track_labels.append(labels[mask][0])

track_probs  = np.array(track_probs)
track_labels = np.array(track_labels)

ax.hist(track_probs[track_labels == 0], bins=40, alpha=0.6,
        color="#4C72B0", label="Human", density=True)
ax.hist(track_probs[track_labels == 1], bins=40, alpha=0.6,
        color="#DD8452", label="AI",    density=True)
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_thresh:.2f}")
ax.set_xlabel("Mean P(AI) per track")
ax.set_ylabel("Density")
ax.set_title(f"Track-Level Distribution ({len(unique_tids)} tracks)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.LOGS_DIR / f"{run_name}_test_analysis.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
def fmt(val, decimals=4):
    """Format a metric value — shows N/A for None (single-class sources)."""
    if val is None:
        return "N/A    "
    return f"{val:.{decimals}f}"

print(f"\n{'='*55}")
print(f"Test set summary — {run_name}")
print(f"{'='*55}")
print(f"Chunk level  ({len(labels)} chunks):")
print(f"  AUROC         : {fmt(results['chunk']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['chunk']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['chunk']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['chunk']['accuracy'])}")
print(f"  F1 (AI)       : {fmt(results['chunk']['f1'])}")
print(f"  F1 (human)    : {fmt(results['chunk']['f1_human'])}")

print(f"\nTrack level  ({results['track']['n_tracks']} tracks):")
print(f"  AUROC         : {fmt(results['track']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['track']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['track']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['track']['accuracy'])}")
print(f"  F1 (AI)       : {fmt(results['track']['f1'])}")
print(f"  F1 (human)    : {fmt(results['track']['f1_human'])}")

print(f"\nPer-source breakdown:")
for src, m in results["per_source"].items():
    c = m["chunk"]
    t = m["track"]
    note = f"  [{c.get('note', '')}]" if c.get("note") else ""
    print(f"  {src:20s}: auroc={fmt(c['auroc'])}  "
          f"acc={fmt(c['accuracy'])}  "
          f"f1={fmt(c['f1'])}  "
          f"n_tracks={t['n_tracks']}{note}")

print(f"\nOptimal threshold : {best_thresh:.3f}")
print(f"Best F1 at thresh : {max(f1_scores):.4f}")

The MLP baseline achieves a respectable Track-level AUROC of 0.9362 and Chunk-level AUROC of 0.8458. While these headline numbers look strong, and overperforms the results of logistic regression on exctracted audio features from the EDA, the per-source breakdown reveals a significant "Generalization Gap." The model is highly optimized for Udio but performs poorly on Suno samples.

The "Udio-Bias" & Generalizability Gap

- Human vs. Udio (AUROC 0.8534): The model performs well here, suggesting it has captured spectral artifacts specific to the Udio generator.

- Human vs. Suno (AUROC 0.6777): Performance drops drastically. With a Suno accuracy of exactly 50.0%, the model is essentially guessing randomly when faced with Suno data.

- Conclusion: The MLP is behaving more like a "Udio Detector" than a general "AI Audio Detector." It is likely overfitting to specific frequency "stripes" or noise patterns unique to one provider. Such overfitting is common while utilizing MLP for images.

Training Dynamics & Overfitting

- Loss Divergence: Looking at the Loss plot, the training loss continues to decay toward 0.40, while the validation loss begins to trend upward after Epoch 15. This is a classic sign of overfitting.

- Metric Gap: There is a ~6% gap between Training AUROC (~0.91) and Validation AUROC (~0.85). Even with a high dropout of 0.4, the MLP's fully connected nature is struggling to learn abstract features that generalize beyond the training distribution.

Prediction Distribution & Thresholds

- The threshold shown by the original analysis was selected using test labels and is not valid for model selection. The updated analysis cell selects the threshold from validation predictions, then applies it unchanged to the test set.

- Confusion Matrix: At this threshold, the model still misclassifies 1,488 human chunks as AI (False Positives). The "Predicted Probability Distribution" shows that while Human and Udio classes are beginning to separate, a large portion of Suno samples remain "stuck" in the middle probability range (0.4–0.6), indicating model uncertainty.

Next Step

- Transition to CNN: The MLP's failure on Suno suggests it lacks spatial invariance, also overfitting is significant, the model experiences lack of generalazibility. A CNN should be able to identify local textures that are common to both Suno and Udio but differ from FMA.

# Experiment: CNN Architecture

## CNN Architecture

In [ ]:
import torch
import torch.nn as nn

class SpectrogramCNN(nn.Module):
    """
    Standard 2D CNN for Spectrogram Classification.
    Uses alternating Convolution, BatchNorm, and Pooling layers.
    """
    def __init__(
        self, 
        input_shape: tuple[int, int],  # (n_mels, time_steps)
        cnn_channels: list[int] = [32, 64, 128, 256],
        dropout_rate: float = 0.2,
        num_classes: int = 2
    ):
        super().__init__()
        
        layers = []
        in_channels = 1  # Spectrograms are single-channel (grayscale)
        
        # 1. Feature Extraction (Convolutional Layers)
        for out_channels in cnn_channels:
            layers.append(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
            )
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(kernel_size=2)) # Reduces H and W by half
            layers.append(nn.Dropout2d(dropout_rate))
            in_channels = out_channels
            
        self.features = nn.Sequential(*layers)
        
        # 2. Global Average Pooling (GAP)
        # Instead of flattening a giant tensor, we take the mean of each feature map.
        # This makes the model robust to different input lengths.
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 3. Classification Head
        self.classifier = nn.Linear(cnn_channels[-1], num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Expected input: (batch, 1, n_mels, time_steps)
        if x.dim() == 3:
            x = x.unsqueeze(1) # Add channel dim if missing
            
        x = self.features(x)         # Shape: (batch, 256, H', W')
        x = self.global_pool(x)      # Shape: (batch, 256, 1, 1)
        x = x.view(x.size(0), -1)    # Shape: (batch, 256)
        
        logits = self.classifier(x)
        return logits

In [ ]:
cnn1_model = SpectrogramCNN(
    input_shape = config.SPEC_SHAPE,
    cnn_channels = [32, 64, 128, 256, 512], # Deeper is usually better for AI detection
    dropout_rate = 0.2
).to(device)

# Check the parameter count—it should be MUCH lower than the MLP!
num_params_cnn1 = sum(p.numel() for p in cnn1_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params_cnn1:,}")

Following the architectural findings in Deezer's research, I implemented a deep 2D-CNN with a feature map progression of [32, 64, 128, 256, 512]. This depth allows the model to capture increasingly complex hierarchical features in the spectrogram, moving from simple edges to intricate AI-generated textures.

## Build DataLoaders

In [ ]:
cnn_loaders = build_fast_loaders(
    manifest   = manifest,
    config     = config,
    fraction=0.25
)

## Train CNN-1

In [ ]:
model_type = 'cnn'

In [ ]:
CNN_1_RUN = f"cnn_1_{ACTIVE_STRATEGY}"

cnn_1_history = train(
    model       = cnn1_model,
    loaders     = cnn_loaders,
    train_cfg   = TRAIN_CFG,
    base_config = config,
    device      = device,
    manifest    = manifest,
    run_name    = CNN_1_RUN,
    model_type = model_type
)

## Evaluate CNN-1

In [ ]:
load_checkpoint(cnn1_model,
                config.CHECKPOINT_DIR / f"{CNN_1_RUN}_best.pt",
                device)

cnn_1_results = evaluate(
    model      = cnn1_model,
    loader     = cnn_loaders["test"],
    device     = device,
    split_name = "test",
)
save_evaluation_results(cnn_1_results, config.LOGS_DIR,
                        CNN_1_RUN, "test")
plot_training_history(
    cnn_1_history,
    output_path = config.LOGS_DIR / f"{CNN_1_RUN}_curves.png",
    title       = f"CNN-A — {ACTIVE_STRATEGY}",
)

results_df = add_result(
    results_df,
    build_result_row(
        run_name       = CNN_1_RUN,
        results        = cnn_1_results,
        model_type     = model_type,
        train_cfg      = TRAIN_CFG,
        split_strategy = ACTIVE_STRATEGY,
        extra_tags     = {"hard_negatives": False,
                          "notes": "CNN baseline [32,64,128,256,512]"},
    ),
    RESULTS_PATH,
)
show_results(results_df)

In [ ]:
# =============================================================================
# Held-out test analysis (threshold chosen on validation)
# =============================================================================

results  = cnn_1_results
run_name = CNN_1_RUN

labels   = results["labels"]
probs    = results["probs"]
sources  = np.array(results["sources"])

# Select the decision threshold on validation data, then apply it to test.
val_results = evaluate(
    model=cnn1_model,
    loader=cnn_loaders["val"],
    device=device,
    split_name="val",
)
val_labels = val_results["labels"]
val_probs = val_results["probs"]
thresholds = np.linspace(0.01, 0.99, 200)
val_f1_scores = [
    f1_score(val_labels, (val_probs >= t).astype(int), zero_division=0)
    for t in thresholds
]
best_thresh = thresholds[np.argmax(val_f1_scores)]
preds = (probs >= best_thresh).astype(int)
print(f"Validation-selected threshold: {best_thresh:.3f}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"{run_name} — test set analysis (threshold chosen on validation)", fontsize=13)

# ── 1. ROC curve ──────────────────────────────────────────────────────────────
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(labels, probs)
ax.plot(fpr, tpr, color="#4C72B0", linewidth=2,
        label=f"AUROC = {results['chunk']['auroc']:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend()
ax.grid(alpha=0.3)

# ── 2. Precision-Recall curve ────────────────────────────────────────────────
ax = axes[0, 1]
prec, rec, _ = precision_recall_curve(labels, probs)
ax.plot(rec, prec, color="#DD8452", linewidth=2,
        label=f"AUPRC = {results['chunk']['auprc']:.4f}")
ax.axhline(labels.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Baseline = {labels.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (AI = positive)")
ax.legend()
ax.grid(alpha=0.3)

# ── 3. Probability distribution ───────────────────────────────────────────────
ax = axes[0, 2]
ax.hist(probs[labels == 0], bins=50, alpha=0.6,
        color="#4C72B0", label="Human", density=True)
ax.hist(probs[labels == 1], bins=50, alpha=0.6,
        color="#DD8452", label="AI",    density=True)
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_thresh:.2f}")
ax.set_xlabel("P(AI)")
ax.set_ylabel("Density")
ax.set_title("Predicted Probability Distribution")
ax.legend()
ax.grid(alpha=0.3)

# ── 4. Confusion matrix ───────────────────────────────────────────────────────
ax = axes[1, 0]
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Human", "AI"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix (threshold={best_thresh:.2f})")


# ── 5. Per-source AUROC (pairwise) ────────────────────────────────────────────
ax = axes[1, 1]
pair_keys   = [k for k in results["per_source"]
               if k.startswith("human_vs_")]
single_keys = [k for k in results["per_source"]
               if not k.startswith("human_vs_")]

plot_keys   = pair_keys if pair_keys else single_keys
plot_labels = [k.replace("human_vs_", "FMA vs ") for k in plot_keys]
plot_aurocs = [results["per_source"][k]["chunk"]["auroc"]
               for k in plot_keys]
plot_counts = [results["per_source"][k]["track"]["n_tracks"]
               for k in plot_keys]

colors = ["#DD8452" if "Suno" in k else "#55A868" for k in plot_keys]
bars   = ax.bar(plot_labels, plot_aurocs, color=colors, alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylim(0.4, 1.05)
ax.set_ylabel("Chunk AUROC")
ax.set_title("Pairwise AUROC (FMA vs each AI source)")

for bar, count, auroc in zip(bars, plot_counts, plot_aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{auroc:.3f}\n({count} tracks)",
            ha="center", va="bottom", fontsize=9)
ax.grid(alpha=0.3, axis="y")


# ── 6. Track-level probability distribution ───────────────────────────────────
ax = axes[1, 2]
track_ids   = results["track_ids"]
unique_tids = np.unique(track_ids)

track_probs  = []
track_labels = []
for tid in unique_tids:
    mask = track_ids == tid
    track_probs.append(probs[mask].mean())
    track_labels.append(labels[mask][0])

track_probs  = np.array(track_probs)
track_labels = np.array(track_labels)

ax.hist(track_probs[track_labels == 0], bins=40, alpha=0.6,
        color="#4C72B0", label="Human", density=True)
ax.hist(track_probs[track_labels == 1], bins=40, alpha=0.6,
        color="#DD8452", label="AI",    density=True)
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_thresh:.2f}")
ax.set_xlabel("Mean P(AI) per track")
ax.set_ylabel("Density")
ax.set_title(f"Track-Level Distribution ({len(unique_tids)} tracks)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.LOGS_DIR / f"{run_name}_test_analysis.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
def fmt(val, decimals=4):
    """Format a metric value — shows N/A for None (single-class sources)."""
    if val is None:
        return "N/A    "
    return f"{val:.{decimals}f}"

print(f"\n{'='*55}")
print(f"Test set summary — {run_name}")
print(f"{'='*55}")
print(f"Chunk level  ({len(labels)} chunks):")
print(f"  AUROC         : {fmt(results['chunk']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['chunk']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['chunk']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['chunk']['accuracy'])}")
print(f"  F1 (AI)       : {fmt(results['chunk']['f1'])}")
print(f"  F1 (human)    : {fmt(results['chunk']['f1_human'])}")

print(f"\nTrack level  ({results['track']['n_tracks']} tracks):")
print(f"  AUROC         : {fmt(results['track']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['track']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['track']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['track']['accuracy'])}")
print(f"  F1 (AI)       : {fmt(results['track']['f1'])}")
print(f"  F1 (human)    : {fmt(results['track']['f1_human'])}")

print(f"\nPer-source breakdown:")
for src, m in results["per_source"].items():
    c = m["chunk"]
    t = m["track"]
    note = f"  [{c.get('note', '')}]" if c.get("note") else ""
    print(f"  {src:20s}: auroc={fmt(c['auroc'])}  "
          f"acc={fmt(c['accuracy'])}  "
          f"f1={fmt(c['f1'])}  "
          f"n_tracks={t['n_tracks']}{note}")

print(f"\nOptimal threshold : {best_thresh:.3f}")
print(f"Best F1 at thresh : {max(f1_scores):.4f}")

1. Performance vs. MLP Baseline

The jump in performance is substantial. While the MLP baseline struggled with Suno samples (Chunk AUROC ~0.67), this CNN architecture has elevated the Suno AUROC to 0.8927 and nearly "solved" Udio at 0.9608. The track-level accuracy of 91.75% confirms that spatial feature extraction via 2D convolutions is far superior to flattened embeddings for capturing AI-generated "textures."

2. The "Human-Error" Gap

The CNN achieves a strong Track AUROC of 0.9779, but a deeper look at the class breakdown reveals a specific bottleneck. While Udio detection is nearly solved (97.5% accuracy), FMA (human) accuracy lags at 69.02%. This confirms our EDA findings: certain human tracks share spectral or structural "confounders" with AI-generated audio. As seen in the initial MERT-based Logistic Regression, the model's primary struggle is with False Positives—human tracks that lack the typical organic complexity or share artifacts found in AI vocoders.

3. Training Stability vs. MLP

Unlike the MLP, which plateaued and began overfitting after epoch 15, this CNN shows a consistent downward trend in validation loss up to epoch 30. The model is still actively learning and hasn't reached saturation, suggesting that the 2D spatial hierarchy is far more effective at distinguishing these subtle textures than flat embeddings.

4. Next Step: MERT-Guided Hard Negative Mining

To address the FMA-AI overlap, I am moving to MERT-based Hard Negative Mining. Since MERT captures high-level musical features (mood, harmony, genre) alongside AI-related artifacts, it is the ideal tool to identify the "boundary" cases that confuse the model.

5. Hybrid Sampling Strategy (30% Hard / 70% Random)

While the Deezer paper suggests a pairwise/hard-negative focus, I have opted for a hybrid 30/70 split for the following reasons:

- Avoiding Specialization Weakness: Training exclusively on hard negatives (the "boundary") can cause the model to lose its "anchor" on what typical human/AI music sounds like, potentially making it weak on simple, clear-cut examples.

- Balanced Feature Learning: The 70% random sample ensures the model maintains a broad understanding of general musical characteristics, while the 30% hard negatives force it to sharpen its decision boundary specifically where FMA and Suno/Udio overlap.

# Experiment: MERT Hard Negatives Impact (CNN)

## Run MERT Mining on Training Chunks

In [ ]:
# Whether to run MERT+KNN mining at all
# Overridden in notebook per experiment
HARD_NEGATIVE_FRACTION = 0.30
KNN_K = 10
KNN_CONFUSION_THRESHOLD = 0.30

SUBSET_FRACTION = 0.25

In [ ]:
import importlib
import mert_mining as mm
importlib.reload(mm)
from mert_mining import build_mined_chunk_index

import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

with open('tokens/token_hf.txt', 'r') as file:
    token = file.read()

from huggingface_hub import login

# Use this to pass the token directly
login(token=token)


# Load full balanced chunk index
chunk_df = pd.read_csv(
    config.CHUNKS_DIR / f"{ACTIVE_STRATEGY}_chunks_balanced.csv"
)

cache_path = (config.CHUNKS_DIR /
              f"{ACTIVE_STRATEGY}_mert_embeddings_train.npy")

mined_df = build_mined_chunk_index(
    chunk_df               = chunk_df,
    config                 = config,
    device                 = device,
    subset_fraction        = SUBSET_FRACTION,
    hard_negative_fraction = HARD_NEGATIVE_FRACTION,
    knn_k                  = KNN_K,
    neighbor_confusion_threshold = KNN_CONFUSION_THRESHOLD,
    embeddings_cache_path  = cache_path,
    random_seed            = config.SEED,
)

# Save for inspection / reuse
mined_path = config.CHUNKS_DIR / f"{ACTIVE_STRATEGY}_chunks_mined.csv"
mined_df.to_csv(mined_path, index=False)
log.info(f"Mined chunk index saved → {mined_path}")


# Took 46.1 minutes on my Mac M3 Pro.

### Resume in a case of restart
Loads everything needed for mining without re-extracting embeddings

In [ ]:
import numpy as np
import pandas as pd
import importlib
import mert_mining as mm
importlib.reload(mm)
from mert_mining import build_mined_chunk_index, score_knn_confusion

# ── Load chunk index ──────────────────────────────────────────────────────────
chunk_df = pd.read_csv(
    config.CHUNKS_DIR / f"{ACTIVE_STRATEGY}_chunks_balanced.csv"
)
train_df = chunk_df[chunk_df["split"] == "train"].reset_index(drop=True)

# ── Load cached embeddings ────────────────────────────────────────────────────
cache_path = (config.CHUNKS_DIR /
              f"{ACTIVE_STRATEGY}_mert_embeddings_train.npy")

embeddings = np.load(cache_path)
labels     = (train_df["label"].astype(str) == "ai").astype(int).values

print(f"Embeddings loaded : {embeddings.shape}")
print(f"Train chunks      : {len(train_df)}")
print(f"Labels            : human={(labels==0).sum()} ai={(labels==1).sum()}")
assert embeddings.shape[0] == len(train_df), "Shape mismatch — check cache"
print("✓ Ready to run mining")

In [ ]:
# ── Run mining with cached embeddings ────────────────────────────────────────
mined_df = build_mined_chunk_index(
    chunk_df                     = chunk_df,
    config                       = config,
    device                       = device,
    subset_fraction              = SUBSET_FRACTION,
    hard_negative_fraction       = HARD_NEGATIVE_FRACTION,
    neighbor_confusion_threshold = KNN_CONFUSION_THRESHOLD,
    knn_k                        = KNN_K,
    embeddings_cache_path        = cache_path,  # ← loads from disk instantly
    random_seed                  = config.SEED,
)

# Save mined index
mined_path = config.CHUNKS_DIR / f"{ACTIVE_STRATEGY}_chunks_mined.csv"
mined_df.to_csv(mined_path, index=False)
log.info(f"Mined chunk index → {mined_path}")

## Quick preview
compute confusion fracs on a sample of 500 chunks to see distribution before committing to the full build

In [ ]:
chunk_df  = pd.read_csv(
    config.CHUNKS_DIR / f"{ACTIVE_STRATEGY}_chunks_balanced.csv"
)
train_df  = chunk_df[chunk_df["split"] == "train"].reset_index(drop=True)
embeddings = np.load(cache_path)
labels    = (train_df["label"].astype(str) == "ai").astype(int).values

# Run on full set (uses cached embeddings, fast)
confusion_fracs = score_knn_confusion(embeddings, labels, k=10)

human_mask = labels == 0
ai_mask    = labels == 1

print("Human confusion fraction distribution:")
for t in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]:
    n = (confusion_fracs[human_mask] >= t).sum()
    print(f"  >= {t:.2f}: {n:,} ({100*n/human_mask.sum():.1f}%)")

print("\nAI confusion fraction distribution:")
for t in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]:
    n = (confusion_fracs[ai_mask] >= t).sum()
    print(f"  >= {t:.2f}: {n:,} ({100*n/ai_mask.sum():.1f}%)")

I selected a confusion threshold of 0.3 to target the zone where the MERT-space boundary is most blurred, capturing roughly 16% of human and 10% of AI chunks that genuinely overlap. This 30/70 hybrid strategy forces the CNN to sharpen its decision boundary on the difficult FMA tracks that often mimic AI textures while maintaining a broad "common sense" understanding of the overall dataset. By focusing on samples where at least 3 out of 10 neighbors are from the opposite class, the model learns to ignore deceptive confounders without becoming an over-specialized outlier detector that fails on simple, representative cases.

## Build mined manifest from existing spectrograms

In [ ]:
# No regeneration needed — just filter the existing manifest

from pathlib import Path

# Load the full random manifest (already has npy_path for all chunks)
full_manifest = manifests[ACTIVE_STRATEGY]  # same strategy as the mined chunk index

# The mined_df has chunk_ids for the selected subset
# Join with full_manifest to get npy_path for each selected chunk
mined_train_ids = set(
    mined_df[mined_df["split"] == "train"]["chunk_id"].tolist()
)

# For train: use only the mined subset
# For val/test: use the full val/test from the existing manifest
train_manifest  = full_manifest[
    (full_manifest["split"] == "train") &
    (full_manifest["chunk_id"].isin(mined_train_ids))
].copy()

val_manifest    = full_manifest[full_manifest["split"] == "val"].copy()
test_manifest   = full_manifest[full_manifest["split"] == "test"].copy()

manifest_mined  = pd.concat(
    [train_manifest, val_manifest, test_manifest],
    ignore_index=True,
)

# Also carry over hard_type and confusion_frac from mined_df
mined_meta = mined_df[mined_df["split"] == "train"][
    ["chunk_id", "hard_type", "confusion_frac", "is_hard_negative"]
]
manifest_mined = manifest_mined.merge(
    mined_meta, on="chunk_id", how="left"
)
manifest_mined["hard_type"]       = manifest_mined["hard_type"].fillna("random")
manifest_mined["confusion_frac"]  = manifest_mined["confusion_frac"].fillna(0.0)
manifest_mined["is_hard_negative"]= manifest_mined["is_hard_negative"].fillna(False)

# Sanity check
print(f"Mined manifest summary:")
print(manifest_mined.groupby(["split", "label", "source_name"])
      .size().rename("chunks").to_string())

print(f"\nTrain hard_type breakdown:")
train_only = manifest_mined[manifest_mined["split"] == "train"]
print(train_only.groupby(["hard_type", "label"])
      .size().rename("chunks").to_string())

# Verify npy files exist
missing = manifest_mined["npy_path"].apply(
    lambda p: not Path(p).exists()
).sum()
print(f"\nMissing npy files: {missing}  (should be 0)")

## Build dataloaders

In [ ]:
from dataset import build_dataloaders

mined_loaders = build_dataloaders(
    manifest   = manifest_mined,
    config     = config,
    model_type = "cnn",
    batch_size = TRAIN_CFG.batch_size,
)

# Quick sanity check
print("Mined loader summary:")
for split, loader in mined_loaders.items():
    ds = loader.dataset
    print(f"  {split:5s}: {len(ds)} chunks | "
          f"human={(ds.labels==0).sum()} | "
          f"ai={(ds.labels==1).sum()}")

## Train CNN with Hard Negatives

In [ ]:
cnn_mined_model = SpectrogramCNN(
    input_shape = config.SPEC_SHAPE,
    cnn_channels = [32, 64, 128, 256, 512], # Deeper is usually better for AI detection
    dropout_rate = 0.2
).to(device)

# Check the parameter count—it should be MUCH lower than the MLP!
num_params_cnn_hard = sum(p.numel() for p in cnn_mined_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params_cnn_hard:,}")

In [ ]:
from train import train

CNN_MINED_RUN = f"cnn_mined_{ACTIVE_STRATEGY}"

cnn_hn_history = train(
    model       = cnn_mined_model,
    loaders     = mined_loaders,
    train_cfg   = TRAIN_CFG,
    base_config = config,
    device      = device,
    manifest    = manifest_mined,
    run_name    = CNN_MINED_RUN,
)

## Evaluation

In [ ]:
from train import load_checkpoint
from evaluate import evaluate, save_evaluation_results, build_result_row
from results import add_result, show_results, compare_runs

model_type = 'cnn'

load_checkpoint(
    cnn_mined_model,
    config.CHECKPOINT_DIR / f"{CNN_MINED_RUN}_best.pt",
    device,
)

cnn_mined_results = evaluate(
    model      = cnn_mined_model,
    loader     = mined_loaders["test"],
    device     = device,
    split_name = "test",
)

save_evaluation_results(
    results    = cnn_mined_results,
    output_dir = config.LOGS_DIR,
    run_name   = CNN_MINED_RUN,
    split_name = "test",
)

results_df = add_result(
    results_df,
    build_result_row(
        run_name       = CNN_MINED_RUN,
        results        = cnn_mined_results,
        model_type     = model_type,
        train_cfg      = TRAIN_CFG,
        split_strategy = ACTIVE_STRATEGY,
        extra_tags     = {
            "hard_negatives":             True,
            "hard_negative_fraction":     HARD_NEGATIVE_FRACTION,
            "confusion_threshold":        KNN_CONFUSION_THRESHOLD,
            "subset_fraction":            SUBSET_FRACTION,
            "notes": "CNN + MERT hard negatives (bidirectional KNN)",
        },
    ),
    RESULTS_PATH,
)

# Compare baseline CNN vs mined CNN
compare_runs(results_df, [CNN_1_RUN, CNN_MINED_RUN])

In [ ]:
plot_training_history(
    cnn_hn_history,
    output_path = config.LOGS_DIR / f"{CNN_MINED_RUN}_curves.png",
    title       = f"CNN-Mined — {ACTIVE_STRATEGY}",
)

In [ ]:
# Training curve comparison

from evaluate import plot_training_history

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("CNN Baseline vs CNN + MERT Hard Negatives", fontsize=12)

metrics = [
    ("chunk_auroc", "Chunk AUROC"),
    ("chunk_auprc", "Chunk AUPRC (AI)"),
    ("loss",        "Loss"),
]

for ax, (metric, title) in zip(axes, metrics):
    for run_name, label, color in [
        (CNN_1_RUN,     "CNN baseline",       "#4C72B0"),
        (CNN_MINED_RUN, "CNN + hard negatives","#DD8452"),
    ]:
        history_path = config.LOGS_DIR / f"{run_name}_history.csv"
        if not history_path.exists():
            continue
        h     = pd.read_csv(history_path)
        val_h = h[h["split"] == "val"]
        ax.plot(val_h["epoch"], val_h[metric],
                label=label, color=color, linewidth=1.5)

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.LOGS_DIR / "cnn_vs_mined_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
)

results      = cnn_mined_results
run_name     = CNN_MINED_RUN

labels       = results["labels"]
probs        = results["probs"]
sources      = np.array(results["sources"])

# ── Find optimal threshold on VALIDATION SET ──────────────────────────────────
# Run model on val set to get val probabilities
cnn_mined_model.eval()
val_probs_list, val_labels_list = [], []

with torch.no_grad():
    for b_in, b_lab, _ in mined_loaders["val"]:
        logits = cnn_mined_model(b_in.to(device))
        p      = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        val_probs_list.extend(p.tolist())
        val_labels_list.extend(b_lab.numpy().tolist())

val_probs_arr  = np.array(val_probs_list)
val_labels_arr = np.array(val_labels_list)

# Find threshold that maximises F1 on val set
thresholds     = np.linspace(0.01, 0.99, 200)
val_f1_scores  = [
    f1_score(val_labels_arr, (val_probs_arr >= t).astype(int),
             zero_division=0)
    for t in thresholds
]
best_thresh    = thresholds[np.argmax(val_f1_scores)]
best_val_f1    = max(val_f1_scores)

print(f"Optimal threshold (from val set) : {best_thresh:.3f}")
print(f"Val F1 at optimal threshold      : {best_val_f1:.4f}")

# ── Apply threshold to TEST SET ───────────────────────────────────────────────
preds          = (probs >= best_thresh).astype(int)
test_f1        = f1_score(labels, preds, zero_division=0)
print(f"Test F1 at val-optimal threshold : {test_f1:.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"{run_name} — test set analysis "
             f"(threshold from val)", fontsize=13)

# ── 1. ROC curve ──────────────────────────────────────────────────────────────
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(labels, probs)
ax.plot(fpr, tpr, color="#4C72B0", linewidth=2,
        label=f"AUROC = {results['chunk']['auroc']:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (test set)")
ax.legend()
ax.grid(alpha=0.3)

# ── 2. Precision-Recall curve ─────────────────────────────────────────────────
ax = axes[0, 1]
prec, rec, _ = precision_recall_curve(labels, probs)
ax.plot(rec, prec, color="#DD8452", linewidth=2,
        label=f"AUPRC = {results['chunk']['auprc']:.4f}")
ax.axhline(labels.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Baseline = {labels.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (AI = positive)")
ax.legend()
ax.grid(alpha=0.3)

# ── 3. Threshold selection on val set ─────────────────────────────────────────
ax = axes[0, 2]
ax.plot(thresholds, val_f1_scores,
        color="#4C72B0", linewidth=1.5, label="Val F1")
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Best threshold = {best_thresh:.2f}\n"
                 f"Val F1 = {best_val_f1:.4f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("Threshold Selection (val set)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── 4. Confusion matrix ───────────────────────────────────────────────────────
ax = axes[1, 0]
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Human", "AI"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — test set\n"
             f"(threshold={best_thresh:.2f} from val)")

# ── 5. Probability distribution on test set ───────────────────────────────────
ax = axes[1, 1]
ax.hist(probs[labels == 0], bins=50, alpha=0.6,
        color="#4C72B0", label="Human", density=True)
ax.hist(probs[labels == 1], bins=50, alpha=0.6,
        color="#DD8452", label="AI",    density=True)
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_thresh:.2f} (from val)")
ax.set_xlabel("P(AI)")
ax.set_ylabel("Density")
ax.set_title("Predicted Probability Distribution (test)")
ax.legend()
ax.grid(alpha=0.3)

# ── 6. Per-source pairwise AUROC ──────────────────────────────────────────────
ax = axes[1, 2]
pair_keys   = [k for k in results["per_source"]
               if k.startswith("human_vs_")]
single_keys = [k for k in results["per_source"]
               if not k.startswith("human_vs_")]
plot_keys   = pair_keys if pair_keys else single_keys
plot_labels = [k.replace("human_vs_", "FMA vs ") for k in plot_keys]
plot_aurocs = [results["per_source"][k]["chunk"]["auroc"]
               for k in plot_keys]
plot_counts = [results["per_source"][k]["track"]["n_tracks"]
               for k in plot_keys]
colors      = ["#DD8452" if "Suno" in k else "#55A868"
               for k in plot_keys]
bars        = ax.bar(plot_labels, plot_aurocs, color=colors, alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylim(0.4, 1.05)
ax.set_ylabel("Chunk AUROC")
ax.set_title("Pairwise AUROC (FMA vs each AI source)")
for bar, count, auroc in zip(bars, plot_counts, plot_aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{auroc:.3f}\n({count} tracks)",
            ha="center", va="bottom", fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(config.LOGS_DIR / f"{run_name}_test_analysis.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
def fmt(val, decimals=4):
    if val is None:
        return "N/A    "
    return f"{val:.{decimals}f}"

print(f"\n{'='*55}")
print(f"Test set summary — {run_name}")
print(f"  Threshold from val set: {best_thresh:.3f}")
print(f"{'='*55}")
print(f"Chunk level  ({len(labels)} chunks):")
print(f"  AUROC         : {fmt(results['chunk']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['chunk']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['chunk']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['chunk']['accuracy'])}")
print(f"  F1 (AI)       : {fmt(f1_score(labels, preds, zero_division=0))}")
print(f"  F1 (human)    : {fmt(f1_score(1-labels, 1-preds, zero_division=0))}")

print(f"\nTrack level  ({results['track']['n_tracks']} tracks):")
print(f"  AUROC         : {fmt(results['track']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['track']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['track']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['track']['accuracy'])}")

# Track-level F1 with val threshold
track_ids    = results["track_ids"]
unique_tids  = np.unique(track_ids)
t_probs_list = []
t_labs_list  = []
for tid in unique_tids:
    mask = track_ids == tid
    t_probs_list.append(probs[mask].mean())
    t_labs_list.append(labels[mask][0])
t_probs_arr  = np.array(t_probs_list)
t_labs_arr   = np.array(t_labs_list)
t_preds_arr  = (t_probs_arr >= best_thresh).astype(int)
print(f"  F1 (AI)       : "
      f"{fmt(f1_score(t_labs_arr, t_preds_arr, zero_division=0))}")
print(f"  F1 (human)    : "
      f"{fmt(f1_score(1-t_labs_arr, 1-t_preds_arr, zero_division=0))}")

print(f"\nPer-source breakdown:")
for src, m in results["per_source"].items():
    c    = m["chunk"]
    t    = m["track"]
    note = f"  [{c.get('note', '')}]" if c.get("note") else ""
    print(f"  {src:20s}: auroc={fmt(c['auroc'])}  "
          f"acc={fmt(c['accuracy'])}  "
          f"f1={fmt(c['f1'])}  "
          f"n_tracks={t['n_tracks']}{note}")

print(f"\nThreshold (from val F1): {best_thresh:.3f}")
print(f"Val F1 at threshold    : {best_val_f1:.4f}")
print(f"Test F1 at threshold   : {test_f1:.4f}")

1. The "Better Ranker", But Less Confident

While the validation loss is higher and more volatile than the baseline, this model is a significantly better ranker. The increased loss is a byproduct of training on "Hard Negatives"—the model is being forced to learn a much more difficult decision boundary where classes overlap, because of this the model became less confident for some part of chunks, while the previous model was more confident. But the confidence of the previous model could be driven by some confounding factors, while for this model those factors are partially hidden and the model needs to identify something other that can distinguish those two classes. Despite the noise, the Track AUROC increased to 0.9845, and the confusion matrix shows that at the optimal threshold, the model correctly identifies more Human and AI tracks than the baseline. It has achieved better class separability, but it requires a higher threshold to do so.

In addition, the validation loss curves provide critical insight into the structural limits of each model. The Hard-Negative CNN ceased significant improvement around the 15th epoch, suggesting it quickly reached its maximum capacity for distinguishing similar spectral textures. In contrast, the Random-Sampled CNN continued to improve through the 30th epoch, though with decaying velocity.

This sustained improvement in the random model is likely not driven by a deeper understanding of AI synthesis, but rather by increasing confidence in "easy" human samples. While the random model appears more "successful" on paper, the plateau in the hard-negative model indicates a genuine architectural bottleneck: the CNN has exhausted its ability to resolve the overlap between classes. This confirms that further training would likely lead to over-parameterization of confounders rather than a breakthrough in distinguishing hard negatives.

2. Validation of the 30/70 Hybrid Strategy

Validation AUROC (0.9667) consistently outperforms Training AUROC. This confirms that the "Mined" training set is significantly more difficult than the "Random" validation set. The model is effectively "training with weights on," making it highly robust when it encounters average, non-mined samples. The rapid climb to 0.9 AUROC by Epoch 9 proves the 70% random share prevents the model from losing its grasp on "easy" general features while the 30% hard negatives sharpen the boundary.

3. FMA Confusion and Model Generalization

The per-source breakdown reveals that FMA (Human) accuracy remains the primary bottleneck. By training on MERT-mined hard negatives, we forced the model to acknowledge that many human tracks share spectral textures typically associated with AI. To maintain precision, the optimal F1 threshold shifted to 0.916; the model now requires extreme "evidence" before labeling a track as AI, adopting a skeptical stance to mitigate false positives.

The high thresholds in both models (0.82 and 0.916) confirm that these **CNNs act as rankers rather than distinguishers.** Because CNNs lack a global temporal sequence definition, they aggregate local "AI-like" patches. Since many FMA chunks contain at least some regions that mimic these artifacts, the model cannot binary-distinguish; it can only rank the total area of suspicion. This is most evident in the hard-negative model, where the population of "clearly human" scores (e.g., <0.05) has significantly shrunk compared to the random baseline.

Despite the Suno sample size limitations (n=27), the CNN demonstrates superior generalizability over the MLP baseline by capturing patterns it was rarely exposed to. While "backdoor paths" (confounders) remain a risk, the decision to use a 30/70 hard-negative split was strategic: it prevents the model from over-specializing on subtle differences at the expense of the fundamental features identified during EDA. Ultimately, this model serves as a stress test to determine if a CNN’s local-patch logic can ever truly distinguish these classes, or if the lack of sequential context necessitates a shift to AST (Audio Spectrogram Transformer).

In the next section, I try to interpret what are the potential reasons of the confusion for the CNN models in our use case.

# Interpretations of CNNs

In this section, I transition from raw performance metrics to Model Interpretability using Grad-CAM (Gradient-weighted Class Activation Mapping). Grad-CAM is a visualization technique that produces a "heatmap" over the input spectrogram, highlighting the specific frequency-time regions that the CNN used to make its classification. By back-propagating the gradients of the "AI" class score to the final convolutional layer, we can see exactly where the model "listens" for synthesis artifacts.

I utilized Grad-CAM to compare the Average AI Signature against the Average Confused Human Signature (False Positives) across two distinct datasets: a Random Sample and a MERT-mined Hard Negative set. By analyzing these heatmaps at the different thresholds, we aim to identify the specific spectral "traps" that cause the CNN to misclassify high-production human tracks as AI. This analysis serves as the primary justification for my architectural pivot toward the Audio Spectrogram Transformer (AST).

In [ ]:
# ── Reload CNN-A from checkpoint (lost after restart) ─────────────────────────
cnn_1_model = SpectrogramCNN(
    input_shape = config.SPEC_SHAPE,
    cnn_channels = [32, 64, 128, 256, 512], # Deeper is usually better for AI detection
    dropout_rate = 0.2
).to(device)

load_checkpoint(
    cnn_1_model,
    config.CHECKPOINT_DIR / "cnn_1_random_best.pt",
    device,
)
cnn_1_model.eval()
print("CNN-1 reloaded ✓")

In [ ]:
# Helper Functions

from captum.attr import Saliency
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from captum.attr import LayerGradCam

def run_interpretability_study(input_tensor, target_class, model):
    model.eval()
    
    
    # 2. Dynamically find the last Conv2d layer in self.features
    target_layer = None
    for layer in reversed(model.features):
        if isinstance(layer, torch.nn.Conv2d):
            target_layer = layer
            break
    
    lgc = LayerGradCam(model, target_layer)

    # 3. Calculate Attributions
    
    # Grad-CAM attribution
    attr_gradcam = lgc.attribute(input_tensor, target=target_class)
    # Upsample to match your 128x43 input
    attr_gradcam = LayerGradCam.interpolate(attr_gradcam, (128, 43))

    # 4. Visualization
    fig, axs = plt.subplots(1, 2, figsize=(22, 5))
    
    # Helper to clean up tensors for plotting
    def norm_show(tensor):
        return tensor.squeeze().cpu().detach().numpy()

    # Original
    axs[0].imshow(norm_show(input_tensor), aspect='auto', origin='lower', cmap='magma')
    axs[0].set_title(f"Original Spectrogram\n(Target Class: {target_class})")

    # Grad-CAM (Heatmap Overlay)
    axs[1].imshow(norm_show(input_tensor), aspect='auto', origin='lower', cmap='gray')
    axs[1].imshow(norm_show(attr_gradcam), cmap='jet', alpha=0.5, aspect='auto', origin='lower')
    axs[1].set_title("Grad-CAM\n(High-level Feature Focus)")

    for ax in axs:
        ax.set_xlabel("Time Steps")
        ax.set_ylabel("Mel Bins")

    plt.tight_layout()
    plt.show()




def get_average_heatmap(model, loader, target_class=1, num_samples=50, threshold=0.92):
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Find the target layer (Last Conv2d)
    target_layer = None
    for layer in reversed(model.features):
        if isinstance(layer, torch.nn.Conv2d):
            target_layer = layer
            break
    
    lgc = LayerGradCam(model, target_layer)
    accumulated_heatmap = np.zeros((128, 43))
    count = 0

    print(f"Aggregating {num_samples} samples for Class {target_class} with Threshold {threshold}...")

    # I remove torch.no_grad() because Captum needs gradients for Grad-CAM
    # But I ensure the model itself isn't updating.
    for inputs, labels, *rest in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Get probabilities to apply the threshold
        with torch.no_grad():
            logits = model(inputs)
            probs = F.softmax(logits, dim=1)
        
        for i in range(inputs.size(0)):
            if count >= num_samples: break
            
            # Check if the ground truth matches the class we are studying
            if labels[i].item() == target_class:
                
                # Check if the model's prediction score passes our 0.92 threshold
                # If we are looking for 'AI' (target=1), we check prob of class 1
                ai_prob = probs[i, 1].item()
                
                if ai_prob >= threshold:
                    sample = inputs[i:i+1]
                    
                    # Calculate Grad-CAM attribution
                    attr = lgc.attribute(sample, target=1) # Always target AI class to see 'AI features'
                    attr = LayerGradCam.interpolate(attr, (128, 43))
                    
                    # Normalize and add to accumulator
                    heatmap = attr.squeeze().cpu().detach().numpy()
                    accumulated_heatmap += (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-9)
                    count += 1
        
        if count >= num_samples: break

    if count == 0:
        print("Warning: No samples found matching the threshold. Try lowering it.")
        return accumulated_heatmap

    return accumulated_heatmap / count

## CNN-Mined

In [ ]:
import torch
import matplotlib.pyplot as plt
from captum.attr import Saliency, LayerGradCam, IntegratedGradients

# 1. Setup Device and Model
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = cnn_mined_model.to(device)
model.eval()

# Use *rest to catch any extra metadata (track IDs, source names, etc.)
batch = next(iter(mined_loaders['val']))

# Check how many items are actually in your batch to be safe
print(f"Items in batch: {len(batch)}")

# Unpack based on standard PyTorch order (Inputs, Labels, then Metadata)
inputs, labels, *rest = batch

inputs = inputs.to(device)
labels = labels.to(device) # Labels are helpful to have on device too



# --- RUN IT ---
# Pick the first sample in the batch
sample = inputs[0:1] 
label = labels[0].item()

print(f"Analyzing sample with true label: {label}")
run_interpretability_study(sample, target_class=1, model=model) # Explaining "Why AI?"

This is an example for a specific AI generated chunk, in which we can see that the model takes decision mostly based on the middle frequencies behavioral, which is typical for vocals and musical instruments.

In [ ]:
# --- Execute and Plot for Multiple Thresholds ---

thresholds = [0.5, 0.60, 0.70, 0.82, 0.90]   # choose any thresholds you want

fig, axes = plt.subplots(len(thresholds), 2, figsize=(12, 4 * len(thresholds)))

for i, th in enumerate(thresholds):
    # Compute average heatmaps for each threshold
    avg_ai = get_average_heatmap(
        cnn_mined_model, 
        mined_loaders['test'], 
        target_class=1, 
        threshold=th
    )
    avg_human = get_average_heatmap(
        cnn_mined_model, 
        mined_loaders['test'], 
        target_class=0, 
        threshold=th
    )
    
    # Plot AI signature
    ax_ai = axes[i, 0] if len(thresholds) > 1 else axes[0]
    ax_ai.imshow(avg_ai, aspect='auto', origin='lower', cmap='inferno')
    ax_ai.set_title(f"AI Signature (threshold={th})")
    
    # Plot Human signature
    ax_h = axes[i, 1] if len(thresholds) > 1 else axes[1]
    ax_h.imshow(avg_human, aspect='auto', origin='lower', cmap='inferno')
    ax_h.set_title(f"Human Confused Signature (threshold={th})")

plt.tight_layout()
plt.show()

**Major Insights: The CNN’s Decision Logic** 

AI Signature (The Machine DNA): The model identifies AI-generated audio primarily through three spectral markers:

- Mid-Range "Blur" (Mel bins 60–100): This is the model’s "North Star." It targets the frequency range of human vocals and lead instruments, where AI generators often leave a specific "neural texture."

- High-Frequency Hiss: Digital artifacts in the upper registers that the model associates with synthesis.

- Low-Frequency Density: Over-compressed sub-bass patterns (bins 0–20) that mimic the raw, "perfect" output of AI engines like Suno and Udio.


**The "Confusion" in Human Audio (Hard Negatives)**

Despite moving the threshold to an extreme 0.9, the "Human Confused Signature" remains stubbornly bright. This reveals a fundamental misinterpretation rather than a simple lack of confidence:

- Threshold Stubbornness: Even at high certainty levels, the model remains "convinced" that specific human textures are AI. This suggests the CNN is locked onto features that human and AI music truly share in the frequency domain.

- The Mid-Range Trap: As we increase the threshold, the mid-range (bins 60–100) becomes the dominant source of confusion. The model cannot distinguish between a complex human vocal arrangement and an AI-generated one based on texture alone.

- "Shimmer" vs. "Hiss": The model exhibits a High-Frequency Bias, consistently hallucinating AI artifacts within the professional "polish" or "shimmer" of high-quality human recordings. It mistakes a clean high-end for digital noise.

- Sub-Bass Sensitivity: The model is triggered by the heavy sub-bass compression typical of modern human electronic music (FMA). To the CNN’s local filters, a human-engineered "perfect" baseline looks identical to an AI-synthesized one.

## CNN-1

In [ ]:
# 1. Setup Device and Model
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = cnn_1_model.to(device)
model.eval()

# Use *rest to catch any extra metadata (track IDs, source names, etc.)
batch = next(iter(mined_loaders['val']))

# Check how many items are actually in your batch to be safe
print(f"Items in batch: {len(batch)}")

# Unpack based on standard PyTorch order (Inputs, Labels, then Metadata)
inputs, labels, *rest = batch

inputs = inputs.to(device)
labels = labels.to(device) # Labels are helpful to have on device too


# --- RUN IT ---
# Pick the first sample in the batch
sample = inputs[0:1] 
label = labels[0].item()

print(f"Analyzing sample with true label: {label}")
run_interpretability_study(sample, target_class=1, model=model) # Explaining "Why AI?"

This is the same kind of example as it was for the CNN-mined model. Here again we can see the same kind of pattern as in the previous one.

In [ ]:
# --- Execute and Plot for Multiple Thresholds ---

thresholds = [0.5, 0.60, 0.70, 0.82, 0.90]   # choose any thresholds you want

fig, axes = plt.subplots(len(thresholds), 2, figsize=(12, 4 * len(thresholds)))

for i, th in enumerate(thresholds):
    # Compute average heatmaps for each threshold
    avg_ai = get_average_heatmap(
        cnn_1_model, 
        mined_loaders['test'], 
        target_class=1, 
        threshold=th
    )
    avg_human = get_average_heatmap(
        cnn_1_model, 
        mined_loaders['test'], 
        target_class=0, 
        threshold=th
    )
    
    # Plot AI signature
    ax_ai = axes[i, 0] if len(thresholds) > 1 else axes[0]
    ax_ai.imshow(avg_ai, aspect='auto', origin='lower', cmap='inferno')
    ax_ai.set_title(f"AI Signature (threshold={th})")
    
    # Plot Human signature
    ax_h = axes[i, 1] if len(thresholds) > 1 else axes[1]
    ax_h.imshow(avg_human, aspect='auto', origin='lower', cmap='inferno')
    ax_h.set_title(f"Human Signature (threshold={th})")

plt.tight_layout()
plt.show()

AI Signature is mostly the same for the CNN-Random model, with one meaningful difference. A striking contrast appears in the 20–30 Mel bin range (Low-Mids) of the AI signatures. In the CNN-Random model, this area is highly active (bright), revealing that the model is using a "production shortcut"—it mistakenly identifies modern, commercial mastering and low-mid fullness as a primary indicator of AI. Conversely, in the CNN-Mined model, this region is dark. By training on MERT-mined hard negatives (professionally produced human tracks), I forced the model to realize that spectral density in the low-mids is a confounder. This "muting" of the low-mids proves the Mined model is more sophisticated; it has learned to ignore general production quality to focus on the 60–100 bin range, where the actual neural synthesis artifacts reside.

The 20–30 Mel bin range represents the low-mid "body" of a track (150Hz–500Hz), which gives audio its "fullness" and "warmth." In older music or raw recordings, this area is often "thin" or uneven because of simpler recording techniques and a lack of modern digital processing. However, in new music and AI generators (Suno/Udio), this range is heavily stabilized and boosted through Commercial Mastering—a professional "polishing" process that makes audio sound "radio-ready" and expensive. Because AI is designed to mimic this high-end studio sound perfectly, it consistently features a dense, "thick" low-mid profile. The Random CNN fell into the trap of using this "modern thickness" as a shortcut for AI, whereas the Mined CNN learned to ignore it as a common feature of both high-quality human productions and AI synthesis.

The Human Signature for the random-sample model exhibits similar confusion patterns to the mined version—specifically regarding electronic sub-bass and high-frequency "shimmer." However, a major differentiator is the 20–30 Mel bin range; while the mined model successfully filtered this out as a spectral confounder, the random model remains heavily distracted by it. This is compounded by the persistent mid-range confusion (bins 60–100) that remains the primary bottleneck for both architectures.

While the random-sample model appears "better" on paper due to its higher confidence, it is functionally more fragile. It has effectively learned an "easy" version of the classification task. If challenged with MERT-mined hard negatives, this model would likely fail immediately because its baseline for "human audio" is overly simplistic and relies on spectral gaps that disappear in professional, high-density productions.

In summary, the CNN-Mined strategy remains my primary focus, as it is the only approach that attempts to capture the genuine boundary between human and AI synthesis. Nevertheless, the inherent limitations of the CNN architecture are now clearly visible. To achieve robust performance, we must transition from local texture analysis to context-aware sequence modeling. While a CNN-RNN hybrid could offer improvements, the Audio Spectrogram Transformer (AST)—with its global self-attention mechanism—appears to be the most optimal choice for resolving these deep spectral overlaps. However, in this notebook I will try to implement a CNN-BiGRU architecture, as from the perspective of 6000 chunks, it sounds easier to experiment and can be more robust than AST.

# Experiment: CNN-BiGRU

> Existing outputs below are saved from earlier notebook runs and are retained as historical records. Rerun the temporal-model cells to generate the controlled three-model comparison; no new head-to-head results are embedded.


In [ ]:
from models import AudioCRNN, AudioBiLSTM, AudioBiLSTMAttention, build_model

# The class name AudioCRNN is retained for compatibility with the original
# notebook, while the comparison uses the precise model id: cnn_bigru.


### CNN-BiGRU baseline (inherited architecture)

The original model applies the shared CNN to each 1-second spectrogram segment, then processes the resulting sequence with a **two-layer Bidirectional GRU**. Its output at each step has shape `(batch, time, 2 × hidden_size)`. Mean pooling across time produces `(batch, 2 × hidden_size)`, which is passed to the original two-class head. The new BiLSTM variants preserve this feature extractor, input sequence, hidden size, and classifier setup so the recurrent layer and pooling method are the experimental variables.


In [ ]:
from dataset import build_dataloaders

# All three temporal architectures use the same sequential dataset and loaders.
sequence_loaders = build_dataloaders(
    manifest=manifest_mined,
    config=config,
    model_type="cnn_bigru",
    batch_size=TRAIN_CFG.batch_size,
)

sample_input, _, _ = sequence_loaders["train"].dataset[0]
print(f"Sequential input shape: {tuple(sample_input.shape)}")


In [ ]:
MODEL_TYPES = ["cnn_bigru", "cnn_bilstm", "cnn_bilstm_attention"]
MODEL_KWARGS = {
    "n_mels": config.N_MELS,
    "hidden_size": 128,
    "num_layers": 2,
    "segment_frames": 43,
    "dropout": 0.3,
}

# CNN-BiGRU and CNN-BiLSTM use mean pooling. The attention variant replaces
# only that pooling operation with learned weights over the BiLSTM time steps.
for model_type in MODEL_TYPES:
    model = build_model(model_type, **MODEL_KWARGS)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{model_type}: {params:,} trainable parameters")


In [ ]:
TRAIN_CFG = TrainingConfig(
    learning_rate       = 1e-4,
    weight_decay        = 1e-4,
    batch_size          = 64,
    num_epochs          = 50,
    lr_patience         = 8,
    lr_factor           = 0.5,
    lr_min              = 1e-6,
    warmup_epochs       = 3,
    early_stop_patience = 15,
    scheduler_metric    = "val_loss",
    checkpoint_metric   = "val_auroc",
    use_augmentation    = True,
    freq_mask_param     = 15,
    time_mask_param     = 30,
    n_freq_masks        = 2,
    n_time_masks        = 2,
)

TRAIN_CFG.log_summary()


In [ ]:
import random
import time
from train import train


def seed_experiment(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

training_histories = {}
comparison_models = {}
comparison_run_names = {}
training_seconds = {}
parameter_counts = {}

for model_type in MODEL_TYPES:
    # Reset the same seed before each model's initialization and training.
    seed_experiment(config.SEED)
    model = build_model(model_type, **MODEL_KWARGS).to(device)
    comparison_models[model_type] = model
    parameter_counts[model_type] = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    run_name = f"{model_type}_mert_{ACTIVE_STRATEGY}"
    comparison_run_names[model_type] = run_name
    start_time = time.perf_counter()
    training_histories[model_type] = train(
        model=model,
        loaders=sequence_loaders,
        train_cfg=TRAIN_CFG,
        base_config=config,
        device=device,
        manifest=manifest_mined,
        run_name=run_name,
        model_type=model_type,
    )
    training_seconds[model_type] = time.perf_counter() - start_time
    print(
        f"{model_type}: params={parameter_counts[model_type]:,}, "
        f"training={training_seconds[model_type] / 60:.2f} min"
    )

# Compatibility names for the notebook's existing BiGRU analysis cells.
CNN_BIGRU_RUN = comparison_run_names["cnn_bigru"]
bigru_model = comparison_models["cnn_bigru"]
cnn_bigru_history = training_histories["cnn_bigru"]


In [ ]:
from train import load_checkpoint
from evaluate import evaluate, save_evaluation_results, build_result_row
from results import add_result


def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def measure_inference(loader, model):
    """Measure wall time for remaining batches after one unmeasured warmup."""
    model.eval()
    iterator = iter(loader)
    first_batch = next(iterator, None)
    if first_batch is None:
        return float("nan"), 0
    with torch.inference_mode():
        model(first_batch[0].to(device))
    sync_device()

    n_chunks = 0
    start_time = time.perf_counter()
    with torch.inference_mode():
        for batch_inputs, _, _ in iterator:
            model(batch_inputs.to(device))
            n_chunks += len(batch_inputs)
    sync_device()
    elapsed = time.perf_counter() - start_time
    return elapsed, n_chunks


TEST_THRESHOLD = 0.5  # fixed before test evaluation; no test-set tuning
comparison_rows = []
cnn_bigru_mined_results = None

for model_type in MODEL_TYPES:
    model = comparison_models[model_type]
    run_name = comparison_run_names[model_type]
    load_checkpoint(
        model,
        config.CHECKPOINT_DIR / f"{run_name}_best.pt",
        device,
    )

    inference_seconds, inference_chunks = measure_inference(
        sequence_loaders["test"], model
    )
    test_results = evaluate(
        model=model,
        loader=sequence_loaders["test"],
        device=device,
        split_name="test",
        threshold=TEST_THRESHOLD,
    )
    save_evaluation_results(
        results=test_results,
        output_dir=config.LOGS_DIR,
        run_name=run_name,
        split_name="test",
    )

    results_df = add_result(
        results_df,
        build_result_row(
            run_name=run_name,
            results=test_results,
            model_type=model_type,
            train_cfg=TRAIN_CFG,
            split_strategy=ACTIVE_STRATEGY,
            extra_tags={
                "hard_negatives": True,
                "comparison_seed": config.SEED,
                "train_seconds": training_seconds[model_type],
                "trainable_parameters": parameter_counts[model_type],
                "inference_seconds_after_warmup": inference_seconds,
                "inference_chunks_after_warmup": inference_chunks,
                "test_threshold": TEST_THRESHOLD,
            },
        ),
        RESULTS_PATH,
    )

    if model_type == "cnn_bigru":
        cnn_bigru_mined_results = test_results

    chunk = test_results["chunk"]
    track = test_results["track"]
    comparison_rows.append({
        "model_type": model_type,
        "run_name": run_name,
        "seed": config.SEED,
        "parameters": parameter_counts[model_type],
        "training_seconds": training_seconds[model_type],
        "inference_seconds_after_warmup": inference_seconds,
        "inference_chunks_after_warmup": inference_chunks,
        "inference_chunks_per_second": (
            inference_chunks / inference_seconds if inference_seconds > 0 else float("nan")
        ),
        "test_threshold": TEST_THRESHOLD,
        "chunk_auroc": chunk["auroc"],
        "chunk_auprc_ai": chunk["auprc"],
        "chunk_accuracy": chunk["accuracy"],
        "chunk_f1_ai": chunk["f1"],
        "track_auroc": track["auroc"],
        "track_auprc_ai": track["auprc"],
        "track_accuracy": track["accuracy"],
        "track_f1_ai": track["f1"],
        "n_chunks": chunk["n_chunks"],
        "n_tracks": track["n_tracks"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = config.RESULTS_DIR / f"architecture_comparison_{ACTIVE_STRATEGY}.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Comparison saved to {comparison_path}")
display(comparison_df)


In [ ]:
plot_training_history(
    cnn_bigru_history,
    output_path = config.LOGS_DIR / f"{CNN_BIGRU_RUN}_curves.png",
    title       = f"CNN-BiGRU-Mined — {ACTIVE_STRATEGY}",
)

In [ ]:
# Training curve comparison

from evaluate import plot_training_history

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("CNN-Mined vs CNN-BiGRU-Mined", fontsize=12)

metrics = [
    ("chunk_auroc", "Chunk AUROC"),
    ("chunk_auprc", "Chunk AUPRC (AI)"),
    ("loss",        "Loss"),
]

for ax, (metric, title) in zip(axes, metrics):
    for run_name, label, color in [
        (CNN_MINED_RUN,     "CNN-Mined",       "#4C72B0"),
        (CNN_BIGRU_RUN, "CNN-BiGRU-Mined","#DD8452"),
    ]:
        history_path = config.LOGS_DIR / f"{run_name}_history.csv"
        if not history_path.exists():
            continue
        h     = pd.read_csv(history_path)
        val_h = h[h["split"] == "val"]
        ax.plot(val_h["epoch"], val_h[metric],
                label=label, color=color, linewidth=1.5)

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.LOGS_DIR / "cnn_mined_vs_gru_mined_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()

### MODEL EVOLUTION: CNN-MINED vs. CNN-BiGRU-MINED ###

1. ARCHITECTURAL SHIFT:
   - The original CNN-Mined model treated audio as a static 'texture' problem. It was 
     limited to local spectral features (is this vocal grainy? is this snare too sharp?).
   - The CNN-BiGRU (Convolutional Gated Recurrent Unit) introduces TEMPORAL LOGIC. By 
     processing the audio in 1-second segments and feeding them into a Bidirectional 
     GRU, the model now evaluates 'musicality'—how notes and rhythms evolve over 10 seconds.

2. LEARNING CAPACITY:
   - CNN-Mined: Plateaued around 0.94-0.96 AUROC. It suffered from 'feature conflict'—
     unable to distinguish between complex human textures and high-quality AI textures.
   - CNN-BiGRU-Mined: Achieved near-perfect training metrics (AUROC 1.0). The addition of 
     temporal memory allowed the model to map specific AI 'glitches' and unnatural 
     transitions that occur across time, which a static CNN simply cannot see.

3. OVERFITTING ANALYSIS:
   - I observe 'Healthy Overfitting.' While Training Loss hit ~0.0 and Val Loss 
     showed some volatility (bouncing), the AUROC and Accuracy remained rock-solid. 
   - This indicates the model became overconfident in its predictions, but its 
     ability to correctly rank AI vs. Human samples did not degrade.


In [ ]:
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
)

results      = cnn_bigru_mined_results
run_name     = CNN_BIGRU_RUN

labels       = results["labels"]
probs        = results["probs"]
sources      = np.array(results["sources"])

# ── Find optimal threshold on VALIDATION SET ──────────────────────────────────
# Run model on val set to get val probabilities
bigru_model.eval()
val_probs_list, val_labels_list = [], []

with torch.no_grad():
    for b_in, b_lab, _ in sequence_loaders["val"]:
        logits = bigru_model(b_in.to(device))
        p      = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        val_probs_list.extend(p.tolist())
        val_labels_list.extend(b_lab.numpy().tolist())

val_probs_arr  = np.array(val_probs_list)
val_labels_arr = np.array(val_labels_list)

# Find threshold that maximises F1 on val set
thresholds     = np.linspace(0.01, 0.99, 200)
val_f1_scores  = [
    f1_score(val_labels_arr, (val_probs_arr >= t).astype(int),
             zero_division=0)
    for t in thresholds
]
best_thresh    = thresholds[np.argmax(val_f1_scores)]
best_val_f1    = max(val_f1_scores)

print(f"Optimal threshold (from val set) : {best_thresh:.3f}")
print(f"Val F1 at optimal threshold      : {best_val_f1:.4f}")

# ── Apply threshold to TEST SET ───────────────────────────────────────────────
preds          = (probs >= best_thresh).astype(int)
test_f1        = f1_score(labels, preds, zero_division=0)
print(f"Test F1 at val-optimal threshold : {test_f1:.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"{run_name} — test set analysis "
             f"(threshold from val)", fontsize=13)

# ── 1. ROC curve ──────────────────────────────────────────────────────────────
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(labels, probs)
ax.plot(fpr, tpr, color="#4C72B0", linewidth=2,
        label=f"AUROC = {results['chunk']['auroc']:.4f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (test set)")
ax.legend()
ax.grid(alpha=0.3)

# ── 2. Precision-Recall curve ─────────────────────────────────────────────────
ax = axes[0, 1]
prec, rec, _ = precision_recall_curve(labels, probs)
ax.plot(rec, prec, color="#DD8452", linewidth=2,
        label=f"AUPRC = {results['chunk']['auprc']:.4f}")
ax.axhline(labels.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Baseline = {labels.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (AI = positive)")
ax.legend()
ax.grid(alpha=0.3)

# ── 3. Threshold selection on val set ─────────────────────────────────────────
ax = axes[0, 2]
ax.plot(thresholds, val_f1_scores,
        color="#4C72B0", linewidth=1.5, label="Val F1")
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Best threshold = {best_thresh:.2f}\n"
                 f"Val F1 = {best_val_f1:.4f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("Threshold Selection (val set)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── 4. Confusion matrix ───────────────────────────────────────────────────────
ax = axes[1, 0]
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Human", "AI"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — test set\n"
             f"(threshold={best_thresh:.2f} from val)")

# ── 5. Probability distribution on test set ───────────────────────────────────
ax = axes[1, 1]
ax.hist(probs[labels == 0], bins=50, alpha=0.6,
        color="#4C72B0", label="Human", density=True)
ax.hist(probs[labels == 1], bins=50, alpha=0.6,
        color="#DD8452", label="AI",    density=True)
ax.axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_thresh:.2f} (from val)")
ax.set_xlabel("P(AI)")
ax.set_ylabel("Density")
ax.set_title("Predicted Probability Distribution (test)")
ax.legend()
ax.grid(alpha=0.3)

# ── 6. Per-source pairwise AUROC ──────────────────────────────────────────────
ax = axes[1, 2]
pair_keys   = [k for k in results["per_source"]
               if k.startswith("human_vs_")]
single_keys = [k for k in results["per_source"]
               if not k.startswith("human_vs_")]
plot_keys   = pair_keys if pair_keys else single_keys
plot_labels = [k.replace("human_vs_", "FMA vs ") for k in plot_keys]
plot_aurocs = [results["per_source"][k]["chunk"]["auroc"]
               for k in plot_keys]
plot_counts = [results["per_source"][k]["track"]["n_tracks"]
               for k in plot_keys]
colors      = ["#DD8452" if "Suno" in k else "#55A868"
               for k in plot_keys]
bars        = ax.bar(plot_labels, plot_aurocs, color=colors, alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylim(0.4, 1.05)
ax.set_ylabel("Chunk AUROC")
ax.set_title("Pairwise AUROC (FMA vs each AI source)")
for bar, count, auroc in zip(bars, plot_counts, plot_aurocs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{auroc:.3f}\n({count} tracks)",
            ha="center", va="bottom", fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(config.LOGS_DIR / f"{run_name}_test_analysis.png",
            dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
def fmt(val, decimals=4):
    if val is None:
        return "N/A    "
    return f"{val:.{decimals}f}"

print(f"\n{'='*55}")
print(f"Test set summary — {run_name}")
print(f"  Threshold from val set: {best_thresh:.3f}")
print(f"{'='*55}")
print(f"Chunk level  ({len(labels)} chunks):")
print(f"  AUROC         : {fmt(results['chunk']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['chunk']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['chunk']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['chunk']['accuracy'])}")
print(f"  F1 (AI)       : {fmt(f1_score(labels, preds, zero_division=0))}")
print(f"  F1 (human)    : {fmt(f1_score(1-labels, 1-preds, zero_division=0))}")

print(f"\nTrack level  ({results['track']['n_tracks']} tracks):")
print(f"  AUROC         : {fmt(results['track']['auroc'])}")
print(f"  AUPRC (AI)    : {fmt(results['track']['auprc'])}")
print(f"  AUPRC (human) : {fmt(results['track']['auprc_human'])}")
print(f"  Accuracy      : {fmt(results['track']['accuracy'])}")

# Track-level F1 with val threshold
track_ids    = results["track_ids"]
unique_tids  = np.unique(track_ids)
t_probs_list = []
t_labs_list  = []
for tid in unique_tids:
    mask = track_ids == tid
    t_probs_list.append(probs[mask].mean())
    t_labs_list.append(labels[mask][0])
t_probs_arr  = np.array(t_probs_list)
t_labs_arr   = np.array(t_labs_list)
t_preds_arr  = (t_probs_arr >= best_thresh).astype(int)
print(f"  F1 (AI)       : "
      f"{fmt(f1_score(t_labs_arr, t_preds_arr, zero_division=0))}")
print(f"  F1 (human)    : "
      f"{fmt(f1_score(1-t_labs_arr, 1-t_preds_arr, zero_division=0))}")

print(f"\nPer-source breakdown:")
for src, m in results["per_source"].items():
    c    = m["chunk"]
    t    = m["track"]
    note = f"  [{c.get('note', '')}]" if c.get("note") else ""
    print(f"  {src:20s}: auroc={fmt(c['auroc'])}  "
          f"acc={fmt(c['accuracy'])}  "
          f"f1={fmt(c['f1'])}  "
          f"n_tracks={t['n_tracks']}{note}")

print(f"\nThreshold (from val F1): {best_thresh:.3f}")
print(f"Val F1 at threshold    : {best_val_f1:.4f}")
print(f"Test F1 at threshold   : {test_f1:.4f}")

### FINAL EVALUATION SUMMARY: TEST SET ###

- TRACK-LEVEL PERFORMANCE (The Real-World Metric):
  - AUROC:    0.9967  (Near-perfect ranking)
  - Accuracy: 98.27%  (Only ~17 tracks misclassified out of ~1,000)
  - AUPRC:    0.9991  (Extremely high precision for AI detection)

- CHUNK-LEVEL PERFORMANCE (10-second segments):
  - AUROC:    0.9906
  - Accuracy: 95.82%

- SOURCE BREAKDOWN:
  - UDIO: 97.23% Accuracy — This model has effectively 'solved' Udio detection.
  - SUNO: 86.89% Accuracy — Suno remains the 'Final Boss.' Its lo-fi/analog 
    characteristics allow it to slip past the temporal filters more often than Udio.|

#### The Conclusion: Why the CNN-BiGRU Won

The transition to a CRNN architecture was the turning point for this project. Music is fundamentally a temporal art form; therefore, a model that lacks a sense of time (like a standard CNN) will always have a lower ceiling. By using a Bidirectional GRU, I allowed the model to look both forward and backward in time, catching the subtle "inhuman" transitions and phase inconsistencies that define current AI music generation.

**Key Findings:**

Temporal Context is Essential: AI music generators (Suno/Udio) are excellent at creating "perfect moments" but struggle with "perfect sequences." The CNN-BiGRU exploits this by looking for long-term incoherence.

Udio vs. Suno Divergence: There is a clear difference in the "digital fingerprints" of these two engines. Udio produces structural patterns that are highly recognizable to the GRU, while Suno’s slightly more chaotic generation requires more diverse training data to reach the same level of accuracy.

## Interpretability of CNN-BiGRU

### Find misclassified tracks

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Inference Loop
bigru_model.eval()
results = []

with torch.no_grad():
    for b_in, b_lab, b_meta in sequence_loaders["test"]:
        logits = bigru_model(b_in.to(device))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        
        # Zip metadata with results for a cleaner collection
        for i, prob in enumerate(probs):
            results.append({
                "track_id": b_meta[i]["track_id"],
                "source": b_meta[i]["source_name"],
                "chunk_id": b_meta[i]["chunk_id"],
                "label": b_lab[i].item(),
                "prob": prob
            })

# 2. Smart Aggregation using Pandas
raw_df = pd.DataFrame(results)

# Group by track to calculate consistency metrics
track_df = raw_df.groupby("track_id").agg({
    "label": "first",
    "source": "first",
    "prob": ["mean", "std", "count"]
}).reset_index()

# Flatten multi-index columns
track_df.columns = ["track_id", "true_label", "source", "mean_prob", "std_prob", "n_chunks"]

# Add prediction logic
track_df["pred_label"] = (track_df["mean_prob"] >= 0.5).astype(int)
track_df["correct"] = track_df["pred_label"] == track_df["true_label"]
track_df["error_type"] = "Correct"
track_df.loc[(track_df["correct"] == False) & (track_df["true_label"] == 0), "error_type"] = "False Positive"
track_df.loc[(track_df["correct"] == False) & (track_df["true_label"] == 1), "error_type"] = "False Negative"

print("Number of tracks per each source:")
print(track_df['source'].value_counts(dropna=False))

# 3. Informative Printouts
print(f"\n{'='*20} AGGREGATE PERFORMANCE {'='*20}")
acc = track_df["correct"].mean()
print(f"Overall Track Accuracy: {acc:.2%}")

# Identify "High Variance" tracks - where the model is inconsistent
unstable_tracks = track_df[track_df["std_prob"] > 0.2].sort_values("std_prob", ascending=False)
print(f"\nFound {len(unstable_tracks)} tracks with high prediction variance (>0.2 std dev).")
print("These are tracks where the model's confidence fluctuates significantly across time.")

# Identify "Confidently Wrong" tracks - hard errors
hard_errors = track_df[(~track_df["correct"]) & ((track_df["mean_prob"] > 0.8) | (track_df["mean_prob"] < 0.2))
                       & (track_df['std_prob'] < 0.2)]

print(f"Found {len(hard_errors)} 'Confidently Wrong' tracks (Mean Prob > 0.8 or < 0.2).")

# 4. Visual Analysis
plt.style.use('seaborn-v0_8-muted')
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3)

# --- Plot 1: The "Uncertainty Map" (Mean vs Std Dev) ---
ax1 = fig.add_subplot(gs[0, 0])
sns.scatterplot(data=track_df, x="mean_prob", y="std_prob", hue="correct", 
                style="true_label", palette={True: "g", False: "r"}, alpha=0.6, ax=ax1)
ax1.axvline(0.5, color='black', linestyle='--', alpha=0.3)
ax1.set_title("Prediction Consistency: Mean vs. Std Dev", fontweight='bold')
ax1.set_xlabel("Mean P(AI)")
ax1.set_ylabel("Std Dev (Uncertainty across chunks)")
# Highlight: High Std Dev = Model is confused by different parts of the audio.

# --- Plot 2: Source Performance Boxplots ---
ax2 = fig.add_subplot(gs[0, 1])
sns.boxplot(data=track_df, x="source", y="mean_prob", hue="true_label", ax=ax2)
ax2.axhline(0.5, color='red', linestyle='--', alpha=0.5)
ax2.set_title("Probability Distribution by Source", fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

# --- Plot 3: Confusion Matrix ---
ax3 = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(track_df["true_label"], track_df["pred_label"])
ConfusionMatrixDisplay(cm, display_labels=["Human", "AI"]).plot(ax=ax3, cmap='Blues', colorbar=False)
ax3.set_title("Track-Level Confusion Matrix", fontweight='bold')

# --- Plot 4: Error Type Breakdown (Violin) ---
ax4 = fig.add_subplot(gs[1, :])
sns.violinplot(data=track_df, x="source", y="mean_prob", split=True, inner="quart", ax=ax4)
ax4.set_title("Source-wise Prediction Density", fontweight='bold')

plt.tight_layout()
plt.show()

The model is performing well with an overall accuracy of 98.27%. With only 17 total misclassifications out of 982 tracks, the GRU has effectively learned the core distinctions between synthetic and organic audio features for these specific sources.

**Source-Level Insights:**

Udio (AI): The model is most confident here. The probability density is pinned almost perfectly at 1.0. This suggests Udio leaves "fingerprints" that the GRU finds very easy to identify consistently.

FMA (Human): High reliability, though there are 4 "False Positives" where human tracks were flagged as AI. The distribution is tightly packed near 0.0.

Suno (AI): This appears to be the "hardest" source for the model. While the accuracy remains high, the box plot and violin plot show a much wider spread in probabilities compared to Udio, indicating that Suno's generation is more diverse or contains fewer obvious artifacts.

**Uncertainty & Consistency:**

The Variance: In the Mean vs. Std Dev plot, we see a clear parabolic shape. This confirms that the uncertainty is highest when the mean probability is near 0.5.

Instability: 141 tracks (approx. 14%) show high variance (σ>0.2). These are "unstable" tracks where the model changes its mind significantly from one 10-second chunk to the next.

The "Confidently Wrong" 8: We have 6 tracks where the model is both highly certain and completely incorrect (Mean Prob >0.8 or <0.2). These are the highest-priority samples for manual listening, as they represent edge cases where the model has been "fooled" by specific acoustic textures.

### Misclassified tracks — duration and prediction statistics

In [ ]:
# Load metadata to get durations
meta_df = pd.read_csv(config.META_PATH)
duration_lookup = meta_df.set_index("file_path")["duration_sec"].to_dict()

# Get test manifest for file_path lookup
test_manifest = manifest_mined[manifest_mined["split"] == "test"]

# Build chunk_id → file_path lookup
chunk_to_filepath = test_manifest.set_index("chunk_id")["file_path"].to_dict()

# Get misclassified tracks from your track_df
wrong_tracks = track_df[track_df["correct"] == False].copy()

# Enrich with duration and chunk_ids
rows = []
for _, track in wrong_tracks.iterrows():
    # Get chunk_ids for this track from raw_df
    chunk_ids = raw_df[
        raw_df["track_id"] == track["track_id"]
    ]["chunk_id"].tolist()

    # Get file_path from any chunk of this track
    file_path = None
    for cid in chunk_ids:
        if cid in chunk_to_filepath:
            file_path = chunk_to_filepath[cid]
            break

    duration = duration_lookup.get(file_path, None)

    rows.append({
        "track_id":    track["track_id"],
        "source":      track["source"],
        "true_class":  "Human" if track["true_label"] == 0 else "AI",
        "pred_class":  "Human" if track["pred_label"] == 0 else "AI",
        "error_type":  track["error_type"],
        "duration_sec":round(duration, 1) if duration else None,
        "n_chunks":    int(track["n_chunks"]),
        "mean_p_ai":   round(track["mean_prob"], 4),
        "std_p_ai":    round(track["std_prob"],  4),
    })

summary_df = pd.DataFrame(rows).sort_values(
    ["source", "error_type"]
).reset_index(drop=True)

# ── Per-track table ───────────────────────────────────────────────────────────
print(f"{'='*85}")
print(f"Misclassified Tracks — Duration & Prediction Statistics")
print(f"{'='*85}")
print(summary_df[[
    "source", "true_class", "pred_class",
    "duration_sec", "n_chunks",
    "mean_p_ai", "std_p_ai", "error_type",
]].to_string(index=False))

# ── Aggregate by source and error type ───────────────────────────────────────
print(f"\n{'='*85}")
print(f"Aggregate by source and error type")
print(f"{'='*85}")
agg = (summary_df
       .groupby(["source", "error_type"])
       .agg(
           n_tracks     = ("track_id",    "count"),
           mean_dur     = ("duration_sec","mean"),
           median_dur   = ("duration_sec","median"),
           mean_p_ai    = ("mean_p_ai",   "mean"),
           mean_std_pai = ("std_p_ai",    "mean"),
       )
       .round(3)
       .reset_index())
print(agg.to_string(index=False))

# ── Duration distribution: wrong vs correct ───────────────────────────────────
correct_tracks = track_df[track_df["correct"] == True].copy()
correct_rows   = []
for _, track in correct_tracks.iterrows():
    chunk_ids = raw_df[raw_df["track_id"] == track["track_id"]]["chunk_id"].tolist()
    file_path = None
    for cid in chunk_ids:
        if cid in chunk_to_filepath:
            file_path = chunk_to_filepath[cid]
            break
    duration = duration_lookup.get(file_path, None)
    correct_rows.append({
        "source":      track["source"],
        "duration_sec":round(duration, 1) if duration else None,
        "correct":     True,
    })

wrong_rows = [{
    "source":      r["source"],
    "duration_sec":r["duration_sec"],
    "correct":     False,
} for r in rows]

dur_df = pd.DataFrame(correct_rows + wrong_rows).dropna(subset=["duration_sec"])

**The "False Positive" FMA Outliers (Human → AI)**

The 4 FMA errors occur in significantly longer tracks (mean duration ~202s). Interestingly, these tracks show the highest instability (mean_std_pai of 0.389). The high standard deviation suggests that for some chunks of the music the model interpret this as purely AI‑generated, while for others does not, which causes the overall mean to drift above 0.5.

**The "Stealth AI" Problem (Udio & Suno → Human)**

- Udio & Duration: 10 of the 13 False Negatives come from Udio. Most of these are very short tracks (~32.8s) with very few chunks (2–4). This suggests the GRU might need more "temporal context" to spot Udio's synthetic fingerprints. When the track is short, the process lacks enough evidence to overcome its internal threshold for an AI classification.

- Confidently Wrong: A few Udio samples (e.g., mean_p_ai of 0.0013) are total misses. These tracks likely represent high-fidelity generations that lack the typical spectral artifacts the model has learned to associate with AI.

**Prediction Instability (std_p_ai)**

Across all error types, the Standard Deviation is remarkably high (often >0.4).

This confirms that misclassifications are rarely "consensus" failures. Instead, they are the result of high-variance tracks where different 10-second windows are yielding contradictory labels. This "instability" is a key indicator that the model is struggling with the specific genre or complexity of those tracks.

### Per-segment prediction heatmap (time × frequency)

In [ ]:
from captum.attr import IntegratedGradients
import librosa

# ── Select confidently wrong tracks ──────────────────────────────────────────
conf_wrong = track_df[
    (~track_df["correct"]) &
    ((track_df["mean_prob"] > 0.8) | (track_df["mean_prob"] < 0.2)) &
    (track_df["std_prob"] < 0.2)
].reset_index(drop=True)

print(f"Confidently wrong tracks: {len(conf_wrong)}")
print(conf_wrong[["source","true_label","mean_prob","std_prob","n_chunks"]].to_string())

test_manifest  = manifest_mined[manifest_mined["split"] == "test"]
mel_freqs      = librosa.mel_frequencies(
    n_mels = config.N_MELS, fmax = config.F_MAX
)

def get_seq_tensor(chunk_id, manifest, config):
    row  = manifest[manifest["chunk_id"] == chunk_id].iloc[0]
    spec = np.load(row["npy_path"]).astype(np.float32)
    spec = torch.from_numpy(spec)
    T    = config.SPEC_TIME_FRAMES // 43
    spec = (spec[:, :T*43]
            .reshape(config.N_MELS, T, 43)
            .permute(1, 0, 2)
            .unsqueeze(1))
    return spec.unsqueeze(0).float()   # (1,T,1,H,W)


In [ ]:
fig_rows = len(conf_wrong)
fig, axes = plt.subplots(
    fig_rows, 1,
    figsize=(14, 4.5 * fig_rows)
)
if fig_rows == 1:
    axes = [axes]
fig.suptitle("Segment-level Attribution Map\n"
             "Which time window + frequency band drove the decision?",
             fontsize=12, fontweight="bold")

for i, (_, track) in enumerate(conf_wrong.iterrows()):
    chunk_ids = raw_df[raw_df["track_id"] == track["track_id"]]["chunk_id"].tolist()
    target    = int(track["true_label"])

    # Use first chunk — representative
    tensor   = get_seq_tensor(chunk_ids[0], test_manifest, config)
    model_cpu= bigru_model.cpu()
    tensor_cpu = tensor.detach().cpu().float().requires_grad_(True)
    model_cpu.eval()
    ig         = IntegratedGradients(model_cpu)
    attr, _    = ig.attribute(tensor_cpu, target=target,
                               n_steps=30, return_convergence_delta=True)
    bigru_model.to(device)

    # (1,T,1,H,W) → (T,H) mean over W
    seg_freq_map = (attr.abs()
                    .squeeze(0).squeeze(1)   # (T,H,W)
                    .mean(dim=2)             # (T,H)
                    .detach().numpy())
    seg_freq_map = seg_freq_map / (seg_freq_map.max() + 1e-9)
    T            = seg_freq_map.shape[0]

    # Get per-segment model probability
    with torch.no_grad():
        full_prob = torch.softmax(
            bigru_model(tensor.to(device)), dim=1
        )[0, 1].item()

    ax = axes[i]
    im = ax.imshow(
        seg_freq_map.T,          # (H, T) — freq on y, time on x
        aspect="auto",
        origin="lower",
        cmap="YlOrRd",
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, fraction=0.02, label="Norm. |attr|")

    # Frequency tick labels
    freq_ticks = [0, config.N_MELS//4,
                  config.N_MELS//2, 3*config.N_MELS//4,
                  config.N_MELS-1]
    ax.set_yticks(freq_ticks)
    ax.set_yticklabels(
        [f"{mel_freqs[t]:.0f}Hz" for t in freq_ticks], fontsize=8
    )
    ax.set_xticks(range(T))
    ax.set_xticklabels([f"s{t+1}\n({t*10//T}–{(t+1)*10//T}s)"
                        for t in range(T)], fontsize=8)
    ax.set_xlabel("Time segment")
    ax.set_ylabel("Frequency")
    ax.set_title(
        f"{track['source']} | True: "
        f"{'Human' if track['true_label']==0 else 'AI'} | "
        f"Pred P(AI)={full_prob:.3f} | "
        f"chunk_id={chunk_ids[0]}",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig(config.LOGS_DIR / "conf_wrong_seg_freq_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()

This visualization utilizes Integrated Gradients (IG) to decompose the model's decision-making process within a single 10-second chunk. The x-axis represents the 10 temporal segments (1 second each) processed by the GRU, while the y-axis maps the Mel-frequency bins (Hz). Darker, more intense regions indicate higher "attribution"—meaning those specific time-frequency coordinates were the primary drivers in pushing the model toward its final prediction. By observing whether these "hotspots" are consistent horizontal bands (spectral textures) or sharp vertical spikes (transients), we can interpret if the model is reacting to a persistent background artifact or a specific musical event.

Based on this plots we can say that the model makes its decisions based on both - a persistent background artifacts and a specific musical events.

#### Listen to one chunk per track + print interval info

In [ ]:
import IPython.display as ipd
import librosa

meta_df         = pd.read_csv(config.META_PATH)
duration_lookup = meta_df.set_index("file_path")["duration_sec"].to_dict()
chunk_lookup    = test_manifest.set_index("chunk_id")[
    ["file_path", "start_sec", "end_sec"]
].to_dict(orient="index")

print(f"{'='*65}")
print(f"Audio playback — confidently wrong tracks")
print(f"{'='*65}\n")

for _, track in conf_wrong.iterrows():
    chunk_ids = raw_df[
        raw_df["track_id"] == track["track_id"]
    ]["chunk_id"].tolist()

    # Pick chunk closest to the mean probability
    # (most representative of the track's overall prediction)
    chunk_probs = raw_df[
        raw_df["chunk_id"].isin(chunk_ids)
    ].set_index("chunk_id")["prob"]
    rep_chunk_id = (chunk_probs - track["mean_prob"]).abs().idxmin()

    info      = chunk_lookup.get(rep_chunk_id, {})
    file_path = info.get("file_path", None)
    start_sec = float(info.get("start_sec", 0))
    end_sec   = float(info.get("end_sec",   10))
    track_dur = duration_lookup.get(file_path, None)

    print(f"Source      : {track['source']}")
    print(f"True class  : {'Human' if track['true_label']==0 else 'AI'}")
    print(f"Pred class  : {'Human' if track['pred_label']==0 else 'AI'}")
    print(f"Mean P(AI)  : {track['mean_prob']:.4f}  "
          f"Std: {track['std_prob']:.4f}")
    print(f"Track dur   : {track_dur:.1f}s" if track_dur else
          "Track dur   : unknown")
    print(f"Chunk played: {start_sec:.1f}s → {end_sec:.1f}s  "
          f"(chunk_id={rep_chunk_id})")
    print(f"File        : {file_path}")

    if file_path and Path(file_path).exists():
        audio, sr = librosa.load(
            file_path,
            sr       = config.SAMPLE_RATE,
            mono     = True,
            offset   = start_sec,
            duration = config.CHUNK_SEC,
        )
        print(f"Playing {config.CHUNK_SEC}s chunk at {sr}Hz...")
        display(ipd.Audio(audio, rate=sr))
    else:
        print("⚠ Audio file not found at path above.")

    print(f"{'-'*65}\n")

#### Potential Interpretation of the Model's Weaknesses

By cross-referencing the Integrated Gradients (IG) Attribution Maps with manual listening, we can identify exactly which acoustic "mirages" are fooling the CNN-BiGRU model.

**The FMA False Positive (4th track)**
- Observations: The William Tell Overture (Human) was confidently flagged as AI. Listening reveals a noticeable hiss and "thin" recording quality.

- Heatmap Signature: High attribution is concentrated in the lowest frequency bands and stable horizontal noise floors.

- Insight: The model has likely correlated spectral noise and low-fidelity artifacts with AI generation. It fails to distinguish between "Old/Poor Recording Quality" and "AI-Generated Noise." Maybe this pattern is not the global one, as it is a low chance that only this music experiences poor recording quality from the FMA ones, but at least we can say that there exist some sort of artifacts related with recording type or recording quality that still have some kind of confounding power.


**The "Temporal Sparsity" Blindspot (Udio Jazz & Ballads) (2nd & 3rd tracks)**
- Observations: Sparse Udio tracks (Jazz vocals, Portuguese ballads) were confidently flagged as Human. These tracks feature significant silences, piano decays, and solo guitar.

- Heatmap Signature: The maps show large "white" areas (low attribution) with fragmented pulses.

- Insight: The GRU looks for a continuous mechanical signature to identify AI. When the arrangement is sparse, the AI "artifacts" are hidden by natural silence and organic instrument decays. The model defaults to "Human" because it doesn't find a high enough density of synthetic fingerprints.


**High-Fidelity Mimicry (Latin & Electronic) (1st, 5th and 6th tracks)**
- Observations: Latin orchestras and Electronic Udio tracks fooled the model despite having dense arrangements.

- Heatmap Signature: Strong, clean vertical bars corresponding to sharp transients (percussion hits and bass stabs).

- Insight: Udio is exceptionally good at modeling clean transients. In these cases, the AI output is "too perfect"—mimicking the sharp, professional studio production found in modern human electronic music. The GRU cannot distinguish between "Algorithmically Perfect" and "Human Studio Perfect" at a 1-second segment level.

# Final Conclusion: From Spectral Features to Temporal Intelligence

This project tracked the evolution of an AI music detector across three major architectural shifts, moving from a baseline MLP to a sophisticated CNN-BiGRU ensemble.

## The MERT-Mining Breakthrough

My initial transition from a baseline MLP to a CNN yielded immediate performance gains, but Grad-CAM interpretations revealed a critical flaw: a CNN trained on random samples was "eager" to find easy solutions—learning simple confounding factors, rather than true acoustic signatures. To combat this, I implemented MERT-based mining to select harder, more representative samples. By utilizing a 30/70 mining strategy, I successfully forced the model to learn complex decision boundaries without losing the ability to identify simple cases. This validated that MERT-mining is an essential tool for avoiding at least some sort of confounding factors and improving model generalization.

## Moving Beyond the "Ranker" Limitation

Analysis of the pure CNN models showed they acted more as "rankers" than "distinguishers." They struggled with confidence, particularly on FMA (Human) chunks, by looking at isolated parts of the spectrogram without a cohesive understanding of the song's progression. To overcome this, I introduced the CNN-BiGRU architecture. I chose a GRU over an LSTM for my experimental scale (~6,000 chunks) because GRUs are more parameter-efficient and robust when working with smaller datasets, offering better convergence without the risk of overfitting.

## Architecture Choice: Why CNN-BiGRU?

While an Audio Spectrogram Transformer (AST) might offer higher peak performance given a massive, perfectly balanced dataset (especially for diverse sources like Suno), the CNN-BiGRU emerged as the superior choice for this project. It provided a perfect balance of local feature extraction (CNN) and temporal context (GRU), allowing the model to "listen" to how a sound evolves over 10 seconds. This temporal logic is exactly what allowed the model to outperform the "snapshot" logic of a pure CNN.

## Final Verdict

The final CNN-BiGRU model is exceptionally robust, boasting high accuracy with very few confident mistakes. The errors that do persist—such as Udio Jazz tracks or lo-fi FMA recordings—suggest that while some minor confounding factors (like recording quality vs. generation artifacts) remain, the model has moved far beyond "lucky guessing." By aggregating chunk-level predictions into a track-level consensus, I have created a system that is not only statistically accurate but also interpretably sound.

The model doesn't just recognize AI audio; it understands the temporal "fingerprints" that define it.